# Poches d'air froid (cold-air pools) du Moyen Atlas central : chaîne ECOSTRESS reproductible

Chaîne de traitement de la température de surface nocturne **ECOSTRESS** (70 m) pour détecter les dépressions fermées
qui fonctionnent comme pièges à froid, modéliser le développement nocturne de l'inversion et en tester les déterminants
et la robustesse. Zone d'étude : 3 291 km² centrés sur trois forêts du Moyen Atlas central (Maroc).

**Deux modes d'utilisation** (cellule 1) :

| Mode | Pour qui | Données | Comptes nécessaires |
|---|---|---|---|
| `autonome` | toute personne | paquet de données publié (Zenodo), téléchargé automatiquement | aucun (Earthdata seulement pour reconstruire le cube) |
| `auteur` | l'auteur | Google Drive de l'auteur | Google (Drive), NASA Earthdata, GEE (lecture unique de la ZET) |

**Sections**

| Section | Contenu | Exécution |
|---|---|---|
| A. Mise en place | configuration, environnement, données d'entrée, grille, fonctions | toujours |
| B. Cube ECOSTRESS | inventaire NASA, cache de neige MOD10A1F, construction ou prolongation du cube | si `RECONSTRUIRE_CUBE = True` |
| C. Analyses | part de nuit, climatologies, dépressions, modèle fond − bords, déterminants, robustesse, stabilité | toujours |
| D. Exemples de cuvettes | cartes et séries pour la publication (PNG, PDF, GeoTIFF, CSV) | si `FIGURES_EXEMPLES = True` |
| E. Paquet de données | préparation de l'archive à déposer sur Zenodo | mode `auteur`, si `PREPARER_PAQUET = True` |

Exécuter les cellules **dans l'ordre**. Runtime CPU standard suffisant (RAM < 4 Go). Les sorties sont écrites dans
`sorties/` (tableaux, GeoPackage, GeoTIFF) et `sorties/figures/`.

**Citation et licences** : données https://doi.org/10.5281/zenodo.23472840 ; code https://github.com/Chahbar-UMI/caps-ecostress-moyen-atlas (licence MIT, citation dans `CITATION.cff`).
Données dérivées : ECOSTRESS et MODIS (NASA, accès libre), ESA WorldCover 2021 (CC BY 4.0),
FABDEM (CC BY-NC-SA 4.0 : usage non commercial, partage dans les mêmes conditions).

## A. Mise en place

In [ ]:
# ============================================================
# 1. CONFIGURATION — seule cellule à modifier
# ============================================================
MODE = 'autonome'            # 'autonome' : données publiées, sans compte ; 'auteur' : Drive et comptes de l'auteur
RECONSTRUIRE_CUBE = False    # True : construit ou prolonge le cube depuis la NASA (compte Earthdata requis)
FIGURES_EXEMPLES = True      # section D : cartes et séries d'exemples de cuvettes
EXPORT_SHP = False           # True : exporte aussi les couches vectorielles en shapefile
PREPARER_PAQUET = False      # section E (mode auteur) : prépare l'archive à déposer sur Zenodo
FORMATS_FIG = ('png', 'pdf') # formats d'export des figures

# Mode autonome : archive publiée sur Zenodo (doi:10.5281/zenodo.23472840)
URL_PAQUET = 'https://zenodo.org/records/23472840/files/CAPS_ECOSTRESS_donnees.zip?download=1'
DOSSIER_AUTONOME = 'CAPS_ECOSTRESS_donnees'      # relatif au dossier de travail

# Mode auteur (Google Drive)
DOSSIER_AUTEUR = '/content/drive/MyDrive/CAP_3forets_climatologies_local'
EXPORTS_AUTEUR = '/content/drive/MyDrive/CAP_3forets_exports'
GEE_PROJET = 'rfm-water'                         # lecture unique de la ZET si le GeoJSON manque
GEE_ZET = 'projects/rfm-water/assets/3forest-zet'

In [ ]:
# ============================================================
# 2. ENVIRONNEMENT — paquets, imports, chemins
# ============================================================
# Réf. : Harris et al. (2020, Nature) — NumPy ; Virtanen et al. (2020, Nat. Methods) — SciPy ;
# pandas development team (Zenodo) ; Hunter (2007, Comput. Sci. Eng.) — matplotlib ;
# van der Walt et al. (2014, PeerJ) — scikit-image ; Jordahl et al. (Zenodo) — geopandas ;
# Bartos (2020, Zenodo) — pysheds ; earthaccess development team (Zenodo).
# Bibliographie complète : cellule « Références » en fin de notebook.
import importlib.util, subprocess, sys, os
paquets = {'rasterio': 'rasterio', 'geopandas': 'geopandas', 'netCDF4': 'netCDF4', 'pysheds': 'pysheds',
           'skimage': 'scikit-image', 'sklearn': 'scikit-learn', 'pyproj': 'pyproj', 'shapely': 'shapely'}
if FIGURES_EXEMPLES:
    paquets['contextily'] = 'contextily'
if RECONSTRUIRE_CUBE:
    paquets.update({'earthaccess': 'earthaccess', 'pyhdf': 'pyhdf'})
a_installer = [p for m, p in paquets.items() if importlib.util.find_spec(m) is None]
if a_installer:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a_installer], check=True)

import json, glob, time, shutil, tempfile, hashlib, zipfile, urllib.request
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.lines import Line2D
import rasterio, netCDF4 as nc
from rasterio.crs import CRS
from rasterio.transform import from_origin, Affine
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.features import rasterize, shapes
from rasterio.windows import from_bounds
from scipy import ndimage
from scipy.signal import fftconvolve
from scipy.stats import mannwhitneyu, spearmanr, wilcoxon, kruskal, norm, f as loi_f
from scipy.spatial import cKDTree
from pyproj import Transformer
import shapely
from shapely.geometry import shape, Point
from shapely.ops import unary_union

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if MODE == 'auteur' and EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

if MODE == 'auteur':
    BASE = DOSSIER_AUTEUR
    _fab = sorted(glob.glob(os.path.join(EXPORTS_AUTEUR, 'CAP_FABDEM_30m_marge5km*.tif')))
    F = {'zet': os.path.join(DOSSIER_AUTEUR, 'ZET_3forets.geojson'),
         'fabdem': _fab[0] if _fab else os.path.join(EXPORTS_AUTEUR, 'CAP_FABDEM_30m_marge5km.tif'),
         'cuv_tpi': os.path.join(EXPORTS_AUTEUR, 'CAP_cuvettes_classes_30m.tif'),
         'microrefuges': os.path.join(EXPORTS_AUTEUR, 'CAP_microrefuges_liste.csv'),
         'foret': os.path.join(DOSSIER_AUTEUR, 'entrees', 'foret_WorldCover2021_70m.tif'),
         'neige': os.path.join(DOSSIER_AUTEUR, 'neige_MOD10A1F_grille_ECO.npz')}
elif MODE == 'autonome':
    BASE = os.path.abspath(DOSSIER_AUTONOME)
    _e = os.path.join(BASE, 'entrees')
    F = {'zet': os.path.join(_e, 'ZET_3forets.geojson'),
         'fabdem': os.path.join(_e, 'FABDEM_30m_marge5km.tif'),
         'cuv_tpi': os.path.join(_e, 'cuvettes_TPI_30m.tif'),
         'microrefuges': os.path.join(_e, 'microrefuges_MODIS_1km.csv'),
         'foret': os.path.join(_e, 'foret_WorldCover2021_70m.tif'),
         'neige': os.path.join(_e, 'neige_MOD10A1F_grille_ECO.npz')}
else:
    raise ValueError("MODE doit valoir 'autonome' ou 'auteur'")
DIR_CUBE = os.path.join(BASE, 'cube_ECOSTRESS')
SORTIES = os.path.join(BASE, 'sorties')
FIGURES = os.path.join(SORTIES, 'figures')
TMP = tempfile.mkdtemp(prefix='caps_')
for d in (os.path.dirname(F['foret']), DIR_CUBE, SORTIES, FIGURES):
    os.makedirs(d, exist_ok=True)
pd.set_option('display.width', 220)
print(f'Mode {MODE} ; Colab : {EN_COLAB} ; données : {BASE}')

In [ ]:
# ============================================================
# 3. PARAMÈTRES SCIENTIFIQUES
# ============================================================
# Réf. : Riggs, Hall & Roman (2019) — MOD10A1F C6.1 (neige) ; fenêtres retenues : mois où la
# neige est la plus probable dans le Moyen Atlas ; hors de ces mois, le drapeau vaut 254 (non évalué).
PROJ_NORD_MAROC = ('+proj=lcc +lat_1=33.3 +lat_0=33.3 +lon_0=-5.4 +k_0=0.999625769 '
                   '+x_0=500000 +y_0=300000 +a=6378249.2 +b=6356515 '
                   '+towgs84=31,146,47,0,0,0,0 +units=m +no_defs')
CRS_M = CRS.from_proj4(PROJ_NORD_MAROC)   # Merchich / Nord Maroc (équivalent EPSG:26191)
EPSG_VECT = 'EPSG:26191'                  # SCR déclaré dans les couches vectorielles
PAS_ECO = 70                              # m, grille ECOSTRESS
RAYON_KM = 5.0                            # disque du fond thermique (anomalie = LST − fond)
N_MIN = 5                                 # dates valides minimales par pixel (climatologies)
MOIS_NEIGE_CLIM = [11, 12, 1, 2, 3, 4]    # mois où la neige est retirée (climatologies)
FORET_MAX = 0.2                           # « découvert » : couvert arboré WorldCover ≤ 20 %

PROF_MIN_M, AIRE_MIN_KM2 = 1.0, 0.01      # dépressions fermées retenues
SEUIL_RESEAU_KM2 = 0.5                    # affichage du réseau d'écoulement
NODATA_DEM = -9999.0

N_FOND_MIN, N_BORD_MIN = 3, 10            # pixels valides par passage (fond, bords)
R_TAMPON, R_ANNEAU_MIN, R_ANNEAU_MAX = 100, 300, 1000   # m : anneau des bords
N_NUITS_MODELE, N_BOOT = 15, 300          # modèle fond − bords
PART_DEBUT, PART_FIN = 0.2, 0.9           # positions de référence dans la nuit
N_FOND_RESOLU = 10                        # test de robustesse : fonds bien résolus
PERIODES_STAB = {'2021–2023': (2021, 2023), '2024–2026': (2024, 2026)}
N_NUITS_PERIODE = 12
SEED = 42
NOMS_MOIS = ['janv.', 'févr.', 'mars', 'avr.', 'mai', 'juin',
             'juil.', 'août', 'sept.', 'oct.', 'nov.', 'déc.']
rng = np.random.default_rng(SEED)

In [ ]:
# ============================================================
# 4. DONNÉES D'ENTRÉE — téléchargement (mode autonome) et contrôle
# ============================================================
# Réf. : Hook & Hulley (2022, LP DAAC) — ECOSTRESS ECO_L2T_LSTE v002 ; Hulley et al. (2022,
# IEEE TGRS) — validation/QC ; Fisher et al. (2020, WRR) — géométrie ISS ; Hawker et al. (2022,
# Zenodo) — FABDEM (CC BY-NC-SA 4.0) ; Gorelick et al. (2017, RSE) — Google Earth Engine.
if MODE == 'autonome' and not os.path.exists(F['zet']):
    if 'XXXXXXX' in URL_PAQUET:
        raise ValueError('Renseigner URL_PAQUET (cellule 1) avec l\'adresse de l\'archive Zenodo.')
    _zip = os.path.join(TMP, 'paquet.zip')
    print('Téléchargement du paquet de données…')
    urllib.request.urlretrieve(URL_PAQUET, _zip)
    with zipfile.ZipFile(_zip) as z:
        z.extractall(os.path.dirname(BASE))
    os.remove(_zip)

if MODE == 'auteur' and not os.path.exists(F['zet']):      # lecture unique dans GEE
    import ee
    ee.Authenticate(); ee.Initialize(project=GEE_PROJET)
    json.dump(ee.FeatureCollection(GEE_ZET).geometry().getInfo(), open(F['zet'], 'w'))

cube_present = bool(glob.glob(os.path.join(DIR_CUBE, 'ECOSTRESS_LST_70m_*.nc')))
etat = [('ZET (GeoJSON)', F['zet'], True), ('FABDEM 30 m', F['fabdem'], True),
        ('Cube ECOSTRESS', DIR_CUBE, not RECONSTRUIRE_CUBE),
        ('Couvert forestier 70 m', F['foret'], False), ('Cuvettes TPI 30 m', F['cuv_tpi'], False),
        ('Microrefuges MODIS', F['microrefuges'], False), ('Cache neige MOD10A1F', F['neige'], False)]
manque = []
for nom, chemin, requis in etat:
    ok = cube_present if nom == 'Cube ECOSTRESS' else os.path.exists(chemin)
    print(f'  {"OK " if ok else "—  "} {nom:24s} {"(requis)" if requis else "(facultatif)"}')
    if requis and not ok:
        manque.append(nom)
if manque:
    raise FileNotFoundError(f'Données requises absentes : {manque}')

In [ ]:
# ============================================================
# 5. ZONE D'ÉTUDE ET GRILLE (Merchich / Nord Maroc, 70 m, marge de 5 km)
# ============================================================
def lire_geojson(f):
    g = json.load(open(f))
    if g.get('type') == 'FeatureCollection':
        return unary_union([shape(ft['geometry']) for ft in g['features']])
    return shape(g['geometry']) if g.get('type') == 'Feature' else shape(g)

zet_ll = lire_geojson(F['zet'])
_vers_m = Transformer.from_crs('EPSG:4326', CRS_M, always_xy=True)
zet_m = shapely.transform(zet_ll, lambda c: np.column_stack(_vers_m.transform(c[:, 0], c[:, 1])))
x0, y0, x1, y1 = zet_m.buffer(RAYON_KM * 1000).bounds
x0, y0 = np.floor(x0 / PAS_ECO) * PAS_ECO, np.floor(y0 / PAS_ECO) * PAS_ECO
x1, y1 = np.ceil(x1 / PAS_ECO) * PAS_ECO, np.ceil(y1 / PAS_ECO) * PAS_ECO
W, H = int((x1 - x0) / PAS_ECO), int((y1 - y0) / PAS_ECO)
T_ECO = from_origin(x0, y1, PAS_ECO, PAS_ECO)
EXT = (x0, x1, y0, y1)
xs_e = x0 + PAS_ECO * (np.arange(W) + 0.5)
ys_e = y1 - PAS_ECO * (np.arange(H) + 0.5)
MASQUE_ZET = rasterize([(zet_m, 1)], out_shape=(H, W), transform=T_ECO, fill=0, dtype='uint8').astype(bool)
lo0, la0, lo1, la1 = zet_ll.bounds
BBOX_RECH = (lo0 - 0.06, la0 - 0.06, lo1 + 0.06, la1 + 0.06)
LON_C, LAT_C = zet_ll.centroid.x, zet_ll.centroid.y
r_ = int(round(RAYON_KM * 1000 / PAS_ECO))
yy_, xx_ = np.mgrid[-r_:r_ + 1, -r_:r_ + 1]
NOYAU = (xx_ ** 2 + yy_ ** 2 <= r_ * r_).astype('float32')
print(f'Grille : {W} × {H} pixels de {PAS_ECO} m ; ZET : {MASQUE_ZET.sum() * PAS_ECO ** 2 / 1e6:.1f} km² '
      f'({MASQUE_ZET.sum()} pixels) ; centre {LON_C:.3f}° / {LAT_C:.3f}°')

In [ ]:
# ============================================================
# 6. FONCTIONS COMMUNES
# ============================================================
# Réf. : Clinton & Gong (2013, RSE) — anomalie LST (LST − fond ~5 km) ; Göttsche & Olesen
# (2001, RSE) & Duan et al. (2012, RSE) — part de nuit/jour ; NOAA Solar Calculation Details
# & Meeus (1991) — lever/coucher du soleil.
def anomalie(L):
    """LST − moyenne sur un disque de RAYON_KM (pixels masqués ignorés ; ≥ 10 % du disque valide)."""
    v = np.isfinite(L)
    s = fftconvolve(np.where(v, L, 0), NOYAU, mode='same')
    c = fftconvolve(v.astype('float32'), NOYAU, mode='same')
    fond = np.where(c > 0.1 * NOYAU.sum(), s / np.maximum(c, 1e-6), np.nan)
    return np.where(v, L - fond, np.nan)

def ras(geom):
    """Rasterisation d'une géométrie sur la grille ECOSTRESS."""
    return rasterize([(geom, 1)], out_shape=(H, W), transform=T_ECO, fill=0, dtype='uint8').astype(bool)

def tracer_contour(ax, geom, **kw):
    for g in getattr(geom, 'geoms', [geom]):
        ax.plot(*g.exterior.xy, **kw)

def lever_coucher(d, lat=LAT_C, lon=LON_C):
    """Lever et coucher du soleil (UTC), algorithme solaire NOAA, horizon à −0,833°."""
    jour = pd.Timestamp(d).normalize()
    g = 2 * np.pi / 365 * (jour.dayofyear - 1)
    eqt = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                    - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    la = np.radians(lat)
    ha = np.degrees(np.arccos(np.cos(np.radians(90.833)) / (np.cos(la) * np.cos(decl))
                              - np.tan(la) * np.tan(decl)))
    base = jour.tz_localize('UTC') if jour.tzinfo is None else jour.tz_convert('UTC')
    return (base + pd.Timedelta(minutes=720 - 4 * (lon + ha) - eqt),
            base + pd.Timedelta(minutes=720 - 4 * (lon - ha) - eqt))

def position(t):
    """Position d'un instant dans la nuit (0 = coucher, 1 = lever) ou dans le jour."""
    t = pd.Timestamp(t).tz_convert('UTC')
    lev, cou = lever_coucher(t.date())
    if lev <= t <= cou:
        return dict(nuit_solaire=0, part_jour_ecoulee=(t - lev) / (cou - lev),
                    part_nuit_ecoulee=np.nan, h_depuis_coucher=np.nan, h_avant_lever=np.nan)
    if t < lev:
        debut, fin = lever_coucher(t.date() - pd.Timedelta(days=1))[1], lev
    else:
        debut, fin = cou, lever_coucher(t.date() + pd.Timedelta(days=1))[0]
    return dict(nuit_solaire=1, part_jour_ecoulee=np.nan, part_nuit_ecoulee=(t - debut) / (fin - debut),
                h_depuis_coucher=(t - debut).total_seconds() / 3600,
                h_avant_lever=(fin - t).total_seconds() / 3600)

def copie_atomique(src, dst):
    shutil.copy(src, dst + '.tmp'); os.replace(dst + '.tmp', dst)

def ecrire_tif(chemin, bandes, noms, transform=None, crs=None):
    transform, crs = transform or T_ECO, crs or CRS_M
    h_, w_ = bandes[0].shape
    tmp = os.path.join(TMP, os.path.basename(chemin))
    with rasterio.open(tmp, 'w', driver='GTiff', height=h_, width=w_, count=len(bandes), dtype='float32',
                       crs=crs, transform=transform, nodata=np.nan, compress='deflate', tiled=True) as d:
        for i, (a, nb) in enumerate(zip(bandes, noms), start=1):
            d.write(np.asarray(a, 'float32'), i); d.set_band_description(i, nb)
    copie_atomique(tmp, chemin); os.remove(tmp)

def noms_courts(cols, imposes=None):
    """Noms de champs shapefile : ASCII, ≤ 10 caractères, uniques."""
    import unicodedata
    imposes, sortie, pris = imposes or {}, {}, set()
    for c in cols:
        if c == 'geometry':
            continue
        a = unicodedata.normalize('NFKD', c.replace('–', '_').replace('−', '_')).encode('ascii', 'ignore').decode()
        s = imposes.get(c, a[:10]); base, k = s, 1
        while s in pris:
            s = f'{base[:8]}{k:02d}'; k += 1
        pris.add(s); sortie[c] = s
    return sortie

def ecrire_vecteurs(nom, couches):
    """GeoPackage (une couche par entrée) ; shapefiles en option (noms de champs ≤ 10 caractères)."""
    chemin = os.path.join(SORTIES, nom + '.gpkg')
    tmp = os.path.join(TMP, nom + '.gpkg')
    if os.path.exists(tmp):
        os.remove(tmp)
    for i, (couche, gdf) in enumerate(couches.items()):
        gdf.set_crs(EPSG_VECT, allow_override=True).to_file(tmp, layer=couche, driver='GPKG',
                                                            mode='w' if i == 0 else 'a')
    copie_atomique(tmp, chemin)
    if EXPORT_SHP:
        d_shp = os.path.join(SORTIES, 'shapefiles'); os.makedirs(d_shp, exist_ok=True)
        for couche, gdf in couches.items():
            nc_ = noms_courts(gdf.columns)
            dtmp = os.path.join(TMP, 'shp_' + couche); shutil.rmtree(dtmp, ignore_errors=True); os.makedirs(dtmp)
            gdf.rename(columns=nc_).set_crs(EPSG_VECT, allow_override=True).to_file(
                os.path.join(dtmp, couche + '.shp'), driver='ESRI Shapefile', encoding='utf-8')
            for f in glob.glob(os.path.join(dtmp, couche + '.*')):
                shutil.copy(f, d_shp)
            pd.DataFrame({'champ_shapefile': list(nc_.values()), 'nom_complet': list(nc_.keys())}) \
              .to_csv(os.path.join(d_shp, couche + '_champs.csv'), index=False)
    return chemin

def ecrire_csv(df_, nom, **kw):
    chemin = os.path.join(SORTIES, nom)
    df_.to_csv(chemin + '.tmp', index=False, **kw); os.replace(chemin + '.tmp', chemin)
    return chemin

def sauver_figure(fig, nom):
    for ext in FORMATS_FIG:
        fig.savefig(os.path.join(FIGURES, f'{nom}.{ext}'), dpi=300 if ext == 'png' else None,
                    bbox_inches='tight')

def modifier_nc(f, fonction):
    """Modifie un NetCDF du cube sur une copie locale, puis la recopie (sûr sur Drive)."""
    local = os.path.join(TMP, os.path.basename(f))
    shutil.copy(f, local)
    with nc.Dataset(local, 'a') as ds:
        fonction(ds)
    copie_atomique(local, f); os.remove(local)

def fichiers_cube():
    return sorted(glob.glob(os.path.join(DIR_CUBE, 'ECOSTRESS_LST_70m_*.nc')))

def lire_meta():
    """Une ligne par passage du cube (fichier, indice, date et métadonnées), triée par date."""
    parts = []
    for f in fichiers_cube():
        with nc.Dataset(f) as ds:
            d = pd.DataFrame({'fichier': f, 'k': np.arange(len(ds['time'])),
                              'utc': pd.to_datetime(np.asarray(ds['time'][:]), unit='s', utc=True)})
            for v in ('nuit', 'heure_utc', 'n_tuiles', 'part_valide_ZET', 'instrument_5_bandes', 'nuit_solaire',
                      'part_nuit_ecoulee', 'h_depuis_coucher', 'h_avant_lever', 'part_jour_ecoulee'):
                if v in ds.variables:
                    d[v] = np.ma.filled(ds[v][:].astype('f8'), np.nan)
        parts.append(d)
    m = pd.concat(parts, ignore_index=True).sort_values('utc', ignore_index=True)
    m['mois'], m['annee'] = m['utc'].dt.month, m['utc'].dt.year
    return m

def ecrire_index():
    m = lire_meta()
    copie = os.path.join(TMP, 'index_passages.csv')
    m.drop(columns=['k']).assign(fichier=m['fichier'].map(os.path.basename)).to_csv(copie, index=False)
    copie_atomique(copie, os.path.join(DIR_CUBE, 'index_passages.csv'))
    return m

## B. Cube ECOSTRESS (facultatif : `RECONSTRUIRE_CUBE = True`)

Construit le cube à partir des archives de la NASA (ECO_L2T_LSTE V002, MOD10A1F V061), ou le **prolonge** avec les
nouveaux passages. Nécessite un compte NASA Earthdata (gratuit). Compter environ 3 h pour la construction complète,
quelques minutes pour une prolongation. Reprise automatique après coupure.

In [ ]:
# ============================================================
# 7. CONNEXION EARTHDATA ET FONCTIONS DE LECTURE (ECOSTRESS, MOD10A1F)
# ============================================================
# Réf. : Hulley & Freepartner (2024, JPL) — règles QC ECOSTRESS (masques nuage/eau) ;
# Hulley et al. (2022, IEEE TGRS) — validation ; Riggs, Hall & Roman (2019) — MOD10A1F ;
# NDSI : Hall et al. (2002, RSE) ; remplissage de nuages : Hall et al. (2010, RSE).
# Seuil NDSI ≥ 0,40 : convention de la Collection 5 (Hall et al., 2002), retenue ici ; en C6.1,
# le guide utilisateur laisse ce seuil au choix de l'utilisateur (Riggs et al., 2019).
if RECONSTRUIRE_CUBE:
    import re
    import earthaccess
    from pyhdf.SD import SD, SDC
    auth = earthaccess.login(strategy='interactive', persist=True)   # mot de passe saisi, jamais écrit
    print('Earthdata :', 'connecté' if auth.authenticated else 'ÉCHEC de connexion')
    DIR_TMP = os.path.join(TMP, 'telechargements'); os.makedirs(DIR_TMP, exist_ok=True)
    COUCHES = {'LST': '_LST.tif', 'QC': '_QC.tif', 'cloud': '_cloud.tif', 'water': '_water.tif'}
    SEUIL_NDSI = 40
    SINU = CRS.from_proj4('+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs')
    TUILE_M, H_T, V_T = 1111950.5197665, 17, 5                       # tuile MODIS h17v05
    T_SINU = Affine(TUILE_M / 2400, 0, -20015109.354 + H_T * TUILE_M,
                    0, -TUILE_M / 2400, 10007554.677 - V_T * TUILE_M)

    def heure(g):
        return pd.Timestamp(g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']).tz_convert('UTC')

    def liens(g):
        urls = g.data_links(access='external')
        d = {k: next((u for u in urls if u.endswith(s)), None) for k, s in COUCHES.items()}
        return d if all(d.values()) else None

    def lire_tuile(fich):
        """LST (°C) d'une tuile ; QC bits 0–1 ≤ 1, ciel clair, hors eau ; reprojetée sur la grille."""
        with rasterio.open(fich['LST']) as s:
            lst = s.read(1, masked=True).astype('float32').filled(np.nan)
            lst = lst * (s.scales[0] or 1) + (s.offsets[0] or 0)
            tr_s, crs_s = s.transform, s.crs
        qc = rasterio.open(fich['QC']).read(1).astype('int64')
        cl = rasterio.open(fich['cloud']).read(1)
        wa = rasterio.open(fich['water']).read(1)
        if np.isfinite(lst).any() and np.nanmedian(lst) > 150:
            lst = lst - 273.15
        ok = ((qc & 3) <= 1) & (cl == 0) & (wa == 0) & (lst > -40) & (lst < 70)
        dst = np.full((H, W), np.nan, 'float32')
        reproject(np.where(ok, lst, np.nan).astype('float32'), dst, src_transform=tr_s, src_crs=crs_s,
                  dst_transform=T_ECO, dst_crs=CRS_M, resampling=Resampling.nearest,
                  src_nodata=np.nan, dst_nodata=np.nan)
        return dst

    def mosaique_passage(inv_, p):
        """Assemble les tuiles d'un passage sur la grille complète."""
        L = np.full((H, W), np.nan, 'float32')
        for g in inv_.loc[inv_['passage'] == p, 'g']:
            u = liens(g)
            if u is None:
                continue
            locaux = [str(f) for f in earthaccess.download(list(u.values()), DIR_TMP)]
            try:
                fich = {k: next(f for f in locaux if f.endswith(s)) for k, s in COUCHES.items()}
                L = np.where(np.isnan(L), lire_tuile(fich), L)
            finally:
                for f in locaux:
                    if os.path.exists(f):
                        os.remove(f)
        return L

    def matin_de(p, mode):
        """Matin de référence pour la neige : le matin suivant pour une nuit, le jour même sinon."""
        t = pd.Timestamp(p)
        return ((t + pd.Timedelta(hours=8)) if mode == 'nuit' else t).normalize().date()

    def lire_neige(fichier):
        sd = SD(fichier, SDC.READ)
        noms = list(sd.datasets().keys())
        nom = (next((n for n in noms if n.startswith('CGF') and 'NDSI_Snow_Cover' in n), None)
               or next(n for n in noms if 'NDSI_Snow_Cover' in n))
        a = sd.select(nom)[:].astype('uint8'); sd.end()
        dst = np.zeros((H, W), 'uint8')
        reproject(((a <= 100) & (a >= SEUIL_NDSI)).astype('uint8'), dst, src_transform=T_SINU, src_crs=SINU,
                  dst_transform=T_ECO, dst_crs=CRS_M, resampling=Resampling.nearest)
        return dst

    def tuiles_neige(d0, d1):
        """URL de la tuile MOD10A1F h17v05 (version la plus récente) pour chaque date."""
        t_n = {}
        for g in earthaccess.search_data(short_name='MOD10A1F', bounding_box=BBOX_RECH,
                                         temporal=(str(d0), str(d1)), count=-1):
            url = next((u for u in g.data_links(access='external') if u.endswith('.hdf')), None)
            m = re.search(r'\.A(\d{7})\.h(\d\d)v(\d\d)\.(\d{3})\.', url or '')
            if m and (int(m.group(2)), int(m.group(3))) == (H_T, V_T):
                d = pd.to_datetime(m.group(1), format='%Y%j').date()
                if d not in t_n or m.group(4) > t_n[d][1]:
                    t_n[d] = (url, m.group(4))
        return t_n
else:
    print('Section B ignorée : le cube existant est utilisé.')

In [ ]:
# ============================================================
# 8. INVENTAIRE DES PASSAGES ET CACHE DE NEIGE (octobre à mai)
# ============================================================
# Réf. : Riggs, Hall & Roman (2019) — MOD10A1F C6.1 (neige quotidienne oct.–mai).
MOIS_NEIGE_CUBE = [10, 11, 12, 1, 2, 3, 4, 5]
if RECONSTRUIRE_CUBE:
    PERIODE_CUBE = ('2018-07-01', pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d'))
    res_c = earthaccess.search_data(short_name='ECO_L2T_LSTE', version='002', bounding_box=BBOX_RECH,
                                    temporal=PERIODE_CUBE, count=-1)
    invc = pd.DataFrame({'g': res_c, 't': [heure(g) for g in res_c]})
    invc['passage'] = invc['t'].dt.floor('h')
    invc['mode'] = np.where((invc['t'].dt.hour >= 20) | (invc['t'].dt.hour < 6), 'nuit', 'jour')
    P = (invc.groupby('passage').agg(t=('t', 'min'), n_tuiles=('g', 'size'), mode=('mode', 'first'))
         .reset_index().sort_values('t'))
    P['annee'] = P['t'].dt.year
    P['matin'] = [matin_de(p, m) for p, m in zip(P['passage'], P['mode'])]
    print(f'Passages {PERIODE_CUBE[0]} -> {PERIODE_CUBE[1]} : {len(P)} ({len(invc)} tuiles)')

    cache = dict(np.load(F['neige'])) if os.path.exists(F['neige']) else {}
    def sauver_cache():
        tmp = os.path.join(TMP, 'neige.npz')
        np.savez_compressed(tmp, **cache); copie_atomique(tmp, F['neige'])
    besoin = sorted({d for d in P['matin'] if d.month in MOIS_NEIGE_CUBE})
    manq = [d for d in besoin if str(d) not in cache]
    print(f'Matins de neige nécessaires : {len(besoin)} ; manquants : {len(manq)}')
    if manq:
        t_n = tuiles_neige(min(manq), max(manq) + pd.Timedelta(days=1))
        for k_, d in enumerate([d for d in manq if d in t_n], start=1):
            f = [str(x) for x in earthaccess.download([t_n[d][0]], DIR_TMP)][0]
            cache[str(d)] = lire_neige(f); os.remove(f)
            if k_ % 10 == 0:
                sauver_cache()
        sauver_cache()
        print(f'Cache neige : {len(cache)} matins')

In [ ]:
# ============================================================
# 9. CONSTRUCTION OU PROLONGATION DU CUBE (NetCDF CF-1.8, un fichier par année)
# ============================================================
# Réf. : Eaton et al. — conventions NetCDF Climate and Forecast (CF-1.8, cfconventions.org).
# LST brute (°C, float32) ; drapeau neige séparé (0 absente, 1 neige, 254 non évaluée de juin à septembre,
# 255 information absente). En cas de reprise : les passages manquants sont ajoutés, même dans une année
# déjà écrite ; un passage sans pixel valide est consigné dans passages_vides.csv et ignoré ensuite ;
# un téléchargement échoué est consigné dans journal_echecs_cube.csv et retenté à l'exécution suivante.
NEIGE_ABS, NEIGE_OUI, NEIGE_NON_EVAL, NEIGE_INCONNUE = 0, 1, 254, 255
COPIE_TOUS = 15

def horodatage(t):
    return int(round(pd.Timestamp(t).timestamp()))

def creer_nc(chemin, an):
    ds = nc.Dataset(chemin, 'w', format='NETCDF4')
    ds.createDimension('time', None); ds.createDimension('y', H); ds.createDimension('x', W)
    vx = ds.createVariable('x', 'f8', ('x',)); vx[:] = xs_e
    vx.standard_name, vx.units = 'projection_x_coordinate', 'm'
    vy = ds.createVariable('y', 'f8', ('y',)); vy[:] = ys_e
    vy.standard_name, vy.units = 'projection_y_coordinate', 'm'
    crs = ds.createVariable('crs', 'i4')
    crs.grid_mapping_name = 'lambert_conformal_conic'
    crs.standard_parallel = 33.3
    crs.longitude_of_central_meridian = -5.4
    crs.latitude_of_projection_origin = 33.3
    crs.false_easting, crs.false_northing = 500000.0, 300000.0
    crs.crs_wkt = CRS_M.to_wkt()
    crs.comment = 'Merchich / Nord Maroc (équivalent EPSG:26191), towgs84 = 31,146,47 : voir crs_wkt'
    vt = ds.createVariable('time', 'f8', ('time',))
    vt.standard_name, vt.units, vt.calendar = 'time', 'seconds since 1970-01-01 00:00:00', 'standard'
    vl = ds.createVariable('LST', 'f4', ('time', 'y', 'x'), zlib=True, complevel=4,
                           chunksizes=(1, H, W), fill_value=np.float32(np.nan))
    vl.standard_name, vl.long_name = 'surface_temperature', 'Température de surface ECOSTRESS (70 m)'
    vl.units, vl.grid_mapping = 'degree_Celsius', 'crs'
    vl.comment = ('ECO_L2T_LSTE V002 ; pixels conservés si QC bits 0–1 ≤ 1, cloud = 0, water = 0 ; '
                  'tuiles d\'un même passage assemblées ; reprojection au plus proche voisin')
    vn = ds.createVariable('neige', 'u1', ('time', 'y', 'x'), zlib=True, complevel=4,
                           chunksizes=(1, H, W), fill_value=np.uint8(NEIGE_INCONNUE))
    vn.long_name, vn.grid_mapping = 'Drapeau neige (MOD10A1F, NDSI ≥ 0,40, matin du passage)', 'crs'
    vn.flag_values = np.array([NEIGE_ABS, NEIGE_OUI, NEIGE_NON_EVAL, NEIGE_INCONNUE], 'u1')
    vn.flag_meanings = 'absente neige non_evaluee_juin_septembre information_absente'
    for nom, typ, ln in [('nuit', 'i1', 'Passage entre 20 h et 06 h UTC = 1 (fenêtre fixe), sinon 0'),
                         ('heure_utc', 'f4', 'Heure UTC du passage (heures décimales)'),
                         ('n_tuiles', 'i2', 'Nombre de tuiles assemblées'),
                         ('part_valide_ZET', 'f4', 'Part de la ZET avec une LST valide'),
                         ('instrument_5_bandes', 'i1', 'Passage antérieur au 15 mai 2019 = 1 (seuil approximatif)')]:
        v = ds.createVariable(nom, typ, ('time',)); v.long_name = ln
    ds.Conventions = 'CF-1.8'
    ds.title = f'Cube ECOSTRESS LST 70 m, ZET 3 forêts (Moyen Atlas, Maroc), année {an}'
    ds.source = ('ECOSTRESS ECO_L2T_LSTE V002 (doi:10.5067/ECOSTRESS/ECO_L2T_LSTE.002) ; '
                 'MODIS MOD10A1F V061 (doi:10.5067/MODIS/MOD10A1F.061)')
    ds.history = f'{pd.Timestamp.now(tz="UTC"):%Y-%m-%d} : création (earthaccess, chaîne locale)'
    ds.limitations = ('Neige évaluée d\'octobre à mai seulement, à 500 m (MOD10A1F) ; '
                      'passages sans aucun pixel valide non stockés')
    return ds

def ecrire_passage(ds, r, L):
    if r['t'].month in MOIS_NEIGE_CUBE:
        fl = cache.get(str(r['matin']))
        neige = fl.astype('uint8') if fl is not None else np.full((H, W), NEIGE_INCONNUE, 'uint8')
    else:
        neige = np.full((H, W), NEIGE_NON_EVAL, 'uint8')
    k = len(ds['time'])
    ds['time'][k] = horodatage(r['t'])
    ds['LST'][k, :, :] = L
    ds['neige'][k, :, :] = neige
    ds['nuit'][k] = 1 if r['mode'] == 'nuit' else 0
    ds['heure_utc'][k] = r['t'].hour + r['t'].minute / 60
    ds['n_tuiles'][k] = r['n_tuiles']
    ds['part_valide_ZET'][k] = float(np.isfinite(L)[MASQUE_ZET].mean())
    ds['instrument_5_bandes'][k] = 1 if r['t'] < pd.Timestamp('2019-05-15', tz='UTC') else 0

if RECONSTRUIRE_CUBE:
    F_VIDES = os.path.join(DIR_CUBE, 'passages_vides.csv')
    F_JOURNAL = os.path.join(DIR_CUBE, 'journal_echecs_cube.csv')
    vides = set(pd.read_csv(F_VIDES)['passage']) if os.path.exists(F_VIDES) else set()
    journal = []
    t0 = time.time()
    for an in sorted(P['annee'].unique()):
        final = os.path.join(DIR_CUBE, f'ECOSTRESS_LST_70m_{an}.nc')
        partiel, local = final + '.partiel', os.path.join(TMP, f'ECOSTRESS_LST_70m_{an}.nc')
        source = partiel if os.path.exists(partiel) else (final if os.path.exists(final) else None)
        deja = set()
        if source:
            with nc.Dataset(source) as ds:
                deja = set(np.round(np.asarray(ds['time'][:])).astype('int64').tolist())
        a_faire = [r for _, r in P[P['annee'] == an].iterrows()
                   if horodatage(r['t']) not in deja and str(r['passage']) not in vides]
        if not a_faire:
            print(f'{an} : à jour ({len(deja)} passages)'); continue
        if source:
            shutil.copy(source, local); ds = nc.Dataset(local, 'a')
        else:
            ds = creer_nc(local, an)
        print(f'{an} : {len(a_faire)} passages à traiter ({len(deja)} déjà écrits)')
        nouveaux = 0
        for r in a_faire:
            try:
                L = mosaique_passage(invc, r['passage'])
            except Exception as ex:
                journal.append({'passage': str(r['passage']), 'erreur': repr(ex)[:200]})
                print(f'   ÉCHEC {r["passage"]} : {repr(ex)[:100]}'); continue
            if not np.isfinite(L).any():
                vides.add(str(r['passage'])); continue
            ecrire_passage(ds, r, L); nouveaux += 1
            if nouveaux % COPIE_TOUS == 0:
                ds.sync(); copie_atomique(local, partiel)
                pd.DataFrame({'passage': sorted(vides)}).to_csv(F_VIDES, index=False)
                print(f'   … {len(ds["time"])} passages, copie de sécurité ({(time.time() - t0) / 60:.0f} min)')
        ds.close()
        copie_atomique(local, final); os.remove(local)
        if os.path.exists(partiel):
            os.remove(partiel)
        pd.DataFrame({'passage': sorted(vides)}).to_csv(F_VIDES, index=False)
    if journal:
        pd.DataFrame(journal).to_csv(F_JOURNAL, index=False)
        print(f'{len(journal)} échecs, retentés à la prochaine exécution : {F_JOURNAL}')
    elif os.path.exists(F_JOURNAL):
        os.remove(F_JOURNAL)
    IDX = ecrire_index()
    print(f'Cube : {len(IDX)} passages ; durée {(time.time() - t0) / 60:.0f} min')

## C. Analyses

Toutes les analyses partent du cube (LST brute par passage) et du MNT FABDEM. Aucun téléchargement, sauf la lecture
de WorldCover si la couche de couvert forestier n'est pas fournie.

In [ ]:
# ============================================================
# 10. LECTURE DU CUBE ET POSITION DE CHAQUE PASSAGE DANS LA NUIT
# ============================================================
# Réf. : Göttsche & Olesen (2001, RSE), Duan et al. (2012, RSE) — part de nuit/jour ;
# Fisher et al. (2020, WRR) — géométrie ISS ; NOAA Solar Calculation Details — lever/coucher.
# Nuit solaire et part de la nuit écoulée (0 = coucher, 1 = lever) calculées au centre de la ZET ;
# ajoutées au cube si elles manquent (ou recalculées après une prolongation).
assert fichiers_cube(), 'Cube absent'
with nc.Dataset(fichiers_cube()[0]) as ds:
    assert len(ds['x']) == W and len(ds['y']) == H and np.allclose(ds['x'][:], xs_e, atol=1) \
        and np.allclose(ds['y'][:], ys_e, atol=1), 'Le cube ne correspond pas à la grille de la ZET'

VARS_NUIT = {'part_nuit_ecoulee': 'Part de la nuit écoulée (0 = coucher, 1 = lever)',
             'h_depuis_coucher': 'Heures écoulées depuis le coucher du soleil',
             'h_avant_lever': 'Heures restantes avant le lever du soleil',
             'part_jour_ecoulee': 'Part du jour écoulée (0 = lever, 1 = coucher)',
             'nuit_solaire': 'Soleil sous l\'horizon au moment du passage = 1'}

def ajouter_part_nuit(ds):
    pos = pd.DataFrame([position(t) for t in pd.to_datetime(np.asarray(ds['time'][:]), unit='s', utc=True)])
    for nom, ln in VARS_NUIT.items():
        typ = 'i1' if nom == 'nuit_solaire' else 'f4'
        v = ds[nom] if nom in ds.variables else ds.createVariable(nom, typ, ('time',))
        v.long_name = ln
        v[:] = pos[nom].fillna(-1 if typ == 'i1' else np.nan).to_numpy().astype(typ)
    ds['part_nuit_ecoulee'].comment = f'Algorithme solaire NOAA au centre de la ZET ({LON_C:.3f}°, {LAT_C:.3f}°)'

for f in fichiers_cube():
    with nc.Dataset(f) as ds:
        a_faire = RECONSTRUIRE_CUBE or 'part_nuit_ecoulee' not in ds.variables
    if a_faire:
        modifier_nc(f, ajouter_part_nuit)
META = ecrire_index()

for d in ('2023-12-21', '2023-06-21'):
    lev, cou = lever_coucher(d)
    print(f'  {d} : lever {lev:%H:%M} UTC, coucher {cou:%H:%M} UTC')
print(f'Cube : {len(META)} passages ; nuits solaires {int((META["nuit_solaire"] == 1).sum())} '
      f'(fenêtre fixe 20 h–06 h : {int((META["nuit"] == 1).sum())}) ; '
      f'« nuit » mais soleil levé : {int(((META["nuit"] == 1) & (META["nuit_solaire"] == 0)).sum())} ; '
      f'« jour » mais soleil couché : {int(((META["nuit"] == 0) & (META["nuit_solaire"] == 1)).sum())}')

N_ = META[META['nuit_solaire'] == 1].copy()
N_['moment'] = pd.cut(N_['part_nuit_ecoulee'], [0, 0.4, 0.6, 1.0], include_lowest=True,
                      labels=['début de nuit', 'milieu', 'fin de nuit']).astype(str)
tab = (N_.pivot_table(index='mois', columns='moment', values='utc', aggfunc='count', fill_value=0)
       .reindex(index=range(1, 13), columns=['début de nuit', 'milieu', 'fin de nuit'], fill_value=0))
tab.insert(0, 'nuits', tab.sum(axis=1)); tab.index = NOMS_MOIS
print('\nNuits solaires par mois et position dans la nuit :'); print(tab.to_string())
print('\nPar année :')
print(N_.pivot_table(index='annee', columns='moment', values='utc', aggfunc='count', fill_value=0).to_string())

fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
ax[0].hist(N_['part_nuit_ecoulee'], bins=20, range=(0, 1), color='#2171b5', edgecolor='white')
ax[0].axvspan(0.6, 1.0, color='#08306b', alpha=0.08)
ax[0].set_xlabel('Part de la nuit écoulée (0 = coucher, 1 = lever)'); ax[0].set_ylabel('Passages')
sc = ax[1].scatter(N_['mois'] + rng.uniform(-0.25, 0.25, len(N_)), N_['part_nuit_ecoulee'],
                   c=N_['annee'], cmap='viridis', s=18)
ax[1].set_xticks(range(1, 13)); ax[1].set_xticklabels(NOMS_MOIS, rotation=45)
ax[1].set_ylabel('Part de la nuit écoulée'); ax[1].set_ylim(0, 1); ax[1].axhspan(0.6, 1.0, color='#08306b', alpha=0.08)
fig.colorbar(sc, ax=ax[1], label='Année')
fig.suptitle('Position des passages ECOSTRESS dans la nuit (zone foncée : fin de nuit)')
plt.tight_layout(); sauver_figure(fig, 'position_passages_dans_la_nuit'); plt.show()

In [ ]:
# ============================================================
# 11. CLIMATOLOGIES MENSUELLES D'ANOMALIE (nuit et jour, neige retirée)
# ============================================================
# Réf. : Clinton & Gong (2013, RSE) — anomalie LST ; Riggs, Hall & Roman (2019) —
# retrait de la neige (nov.–avr).
# Anomalie = LST − moyenne du disque de 5 km, passage par passage ; nuit et jour selon la fenêtre fixe
# 20 h–06 h UTC (définition des climatologies publiées) ; neige retirée de novembre à avril avant le fond.
MODES = ['nuit', 'jour']
ACC = {(m, mm): [np.zeros((H, W)), np.zeros((H, W), 'int16'), np.zeros((H, W), 'int16')]
       for m in MODES for mm in range(1, 13)}
t0 = time.time()
for f, g in META.groupby('fichier'):
    with nc.Dataset(f) as ds:
        for _, r in g.iterrows():
            k = int(r['k'])
            L = np.ma.filled(ds['LST'][k, :, :], np.nan).astype('float32')
            S = np.ma.filled(ds['neige'][k, :, :], 255)
            somme, n, nn = ACC[('nuit' if r['nuit'] == 1 else 'jour', int(r['mois']))]
            if r['mois'] in MOIS_NEIGE_CLIM:
                ne = (S == 1) & np.isfinite(L); nn[ne] += 1; L[ne] = np.nan
            A = anomalie(L); v = np.isfinite(A)
            somme[v] += A[v]; n[v] += 1
print(f'Climatologies calculées en {(time.time() - t0) / 60:.1f} min')

clim = {k: (np.where((v[1] >= N_MIN) & MASQUE_ZET, v[0] / np.maximum(v[1], 1), np.nan).astype('float32'),
            v[1], v[2]) for k, v in ACC.items()}

def anom_periode(mode, mois):
    s = sum(ACC[(mode, mm)][0] for mm in mois); n = sum(ACC[(mode, mm)][1].astype('int32') for mm in mois)
    return np.where((n >= N_MIN) & MASQUE_ZET, s / np.maximum(n, 1), np.nan)

bilan = pd.DataFrame([{'mode': m, 'mois': NOMS_MOIS[mm - 1],
                       f'ZET ≥ {N_MIN} dates (%)': round(100 * ((clim[(m, mm)][1] >= N_MIN) & MASQUE_ZET).sum()
                                                       / MASQUE_ZET.sum()),
                       'dates (médiane ZET)': int(np.median(clim[(m, mm)][1][MASQUE_ZET])),
                       'retraits neige (moy. ZET)': round(float(clim[(m, mm)][2][MASQUE_ZET].mean()), 1)}
                      for m in MODES for mm in range(1, 13)])
print(bilan.to_string(index=False))
ecrire_csv(bilan, 'climatologies_bilan.csv')

In [ ]:
# ============================================================
# 12. COUVERT FORESTIER, CARTES ET GeoTIFF DES CLIMATOLOGIES, TABLEAU À COUVERT ÉGAL
# ============================================================
# Réf. : Zanaga et al. (2022, Zenodo) — ESA WorldCover 10 m 2021 v200 (CC BY 4.0 ; classe 10).
# Couvert : part de couvert arboré ESA WorldCover 2021 (classe 10) par pixel de 70 m. Lu dans la couche fournie,
# sinon calculé depuis le dépôt public ESA puis enregistré dans entrees/.
if os.path.exists(F['foret']):
    with rasterio.open(F['foret']) as s:
        FRAC_FORET = s.read(1).astype('float32')
else:
    WC_URLS = [f'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/'
               f'ESA_WorldCover_10m_2021_v200_{t}_Map.tif' for t in ('N30W006', 'N33W006')]
    acc_f, acc_v = np.zeros((H, W), 'float32'), np.zeros((H, W), 'float32')
    for url in WC_URLS:
        with rasterio.open(url) as s:
            win = from_bounds(*BBOX_RECH, transform=s.transform).round_offsets().round_lengths()
            a = s.read(1, window=win, boundless=True, fill_value=0)
            for src, acc in [((a == 10).astype('float32'), acc_f), ((a > 0).astype('float32'), acc_v)]:
                tmp = np.zeros((H, W), 'float32')
                reproject(src, tmp, src_transform=s.window_transform(win), src_crs=s.crs, dst_transform=T_ECO,
                          dst_crs=CRS_M, resampling=Resampling.average)
                acc += tmp
    FRAC_FORET = np.where(acc_v > 0, acc_f / np.maximum(acc_v, 1e-6), np.nan).astype('float32')
    ecrire_tif(F['foret'], [FRAC_FORET], ['part_couvert_arbore'])
OUVERT = np.nan_to_num(FRAC_FORET, nan=1.0) <= FORET_MAX
print(f'Couvert arboré moyen de la ZET : {np.nanmean(FRAC_FORET[MASQUE_ZET]):.0%}')

CUV30 = FRAC_CUV = None                                     # cuvettes TPI (facultatif, superposition)
if os.path.exists(F['cuv_tpi']):
    with rasterio.open(F['cuv_tpi']) as s:
        CUV30 = np.isin(s.read(1), [1, 2]).astype('float32')
        xs30 = s.bounds.left + s.res[0] * (np.arange(s.width) + 0.5)
        ys30 = s.bounds.top - s.res[1] * (np.arange(s.height) + 0.5)
        FRAC_CUV = np.zeros((H, W), 'float32')
        reproject(CUV30, FRAC_CUV, src_transform=s.transform, src_crs=s.crs, dst_transform=T_ECO,
                  dst_crs=CRS_M, resampling=Resampling.average)

blanc = ListedColormap(['white']); blanc.set_bad(alpha=0)
hors_zet = np.where(MASQUE_ZET, np.nan, 1.0)
for mode in MODES:
    tout = np.concatenate([clim[(mode, mm)][0][np.isfinite(clim[(mode, mm)][0])] for mm in range(1, 13)])
    v = float(np.percentile(np.abs(tout), 98))
    cmap = plt.get_cmap('RdBu_r').copy(); cmap.set_bad('#cfcfcf')
    fig, axes = plt.subplots(3, 4, figsize=(17, 13))
    for mm, ax in zip(range(1, 13), axes.ravel()):
        moy, n, _ = clim[(mode, mm)]
        im = ax.imshow(moy, cmap=cmap, vmin=-v, vmax=v, extent=EXT)
        ax.imshow(hors_zet, cmap=blanc, extent=EXT)
        if CUV30 is not None:
            ax.contour(xs30, ys30, CUV30, levels=[0.5], colors='k', linewidths=0.25)
        ax.contour(xs_e, ys_e, np.nan_to_num(FRAC_FORET), levels=[0.5], colors='#1b7837', linewidths=0.3)
        tracer_contour(ax, zet_m, color='k', lw=0.8)
        ax.set_title(f'{NOMS_MOIS[mm - 1]} — {int(np.median(n[MASQUE_ZET]))} dates', fontsize=10)
        ax.set_axis_off()
    fig.colorbar(im, ax=axes, shrink=0.55, label='Anomalie de LST (°C) : pixel − fond 5 km')
    fig.suptitle(f'ECOSTRESS {mode}, climatologie mensuelle (neige retirée) — vert : forêt ; '
                 f'gris : < {N_MIN} dates' + ('; noir : cuvettes TPI' if CUV30 is not None else ''), fontsize=12)
    sauver_figure(fig, f'climatologie_ECOSTRESS_{mode}'); plt.show()
    bandes, noms = [], []
    for nom_b, i in [('anom', 0), ('n', 1), ('n_neige', 2)]:
        for mm in range(1, 13):
            bandes.append(np.where(MASQUE_ZET, clim[(mode, mm)][i], np.nan)); noms.append(f'{nom_b}_m{mm:02d}')
    ecrire_tif(os.path.join(SORTIES, f'climatologie_ECOSTRESS_{mode}_70m.tif'), bandes, noms)

if FRAC_CUV is not None:                                    # tableau à couvert égal (cuvettes TPI)
    cuv_dec = MASQUE_ZET & (FRAC_CUV >= 0.5) & OUVERT
    hors_dec = MASQUE_ZET & (FRAC_CUV == 0) & OUVERT
    foret = MASQUE_ZET & (np.nan_to_num(FRAC_FORET) >= 0.7)
    lig = [{'mode': m, 'mois': NOMS_MOIS[mm - 1], 'cuvette découverte': np.nanmean(clim[(m, mm)][0][cuv_dec]),
            'découvert hors cuvette': np.nanmean(clim[(m, mm)][0][hors_dec]),
            'forêt': np.nanmean(clim[(m, mm)][0][foret])} for m in MODES for mm in range(1, 13)]
    T = pd.DataFrame(lig); T['écart cuvette − hors'] = T['cuvette découverte'] - T['découvert hors cuvette']
    print(T.round(2).to_string(index=False)); ecrire_csv(T.round(3), 'climatologies_couvert_egal.csv')

In [ ]:
# ============================================================
# 13. DÉPRESSIONS FERMÉES ET ÉCOULEMENT POTENTIEL DE L'AIR FROID (FABDEM)
# ============================================================
# Réf. : Vincent (1993, IEEE TIP) — reconstruction morphologique ; van der Walt et al. (2014,
# PeerJ) — scikit-image ; O'Callaghan & Mark (1984) — D8 ; Barnes et al. (2014a, Comput.
# Geosci. 117–127) — famille Priority-flood (facultatif : fill_pits ne comble que les creux d'un
# pixel) ; Barnes et al. (2014b, 128–135) — drainage des surfaces planes (resolve_flats) ;
# Bartos (2020, Zenodo) — pysheds ; pièges à froid : Clements et al. (2003), Whiteman et al.
# (2004), Steinacker et al. (2007), Pastore et al. (2024).
# Dépressions : reconstruction morphologique (bords et nodata = exutoires), profondeur ≥ PROF_MIN_M,
# surface ≥ AIRE_MIN_KM2. Écoulement : D8 sans comblement des dépressions (l'air froid s'y arrête).
if not hasattr(np, 'in1d'):                  # compatibilité pysheds / NumPy ≥ 2.4 (np.in1d retiré)
    np.in1d = lambda a, b, **kw: np.isin(np.ravel(a), b, **kw)
from pysheds.grid import Grid
from skimage.morphology import reconstruction

with rasterio.open(F['fabdem']) as s:
    z = s.read(1).astype('float32'); crs_fab, tr_fab, meta_fab = s.crs, s.transform, s.profile
sh_fab, pas_fab = z.shape, abs(tr_fab.a)
invalide = ~np.isfinite(z) | (z <= 0)
z_net = np.where(invalide, NODATA_DEM, z).astype('float32')
F_FAB_NET = os.path.join(TMP, 'fabdem_net.tif')
meta_fab.update(dtype='float32', nodata=NODATA_DEM, count=1, compress='deflate')
with rasterio.open(F_FAB_NET, 'w', **meta_fab) as d:
    d.write(z_net, 1)

z_rec = np.where(invalide, z[~invalide].min() - 100, z).astype('float64')
bord = np.zeros(sh_fab, bool); bord[[0, -1], :] = True; bord[:, [0, -1]] = True
exut = bord | invalide | ndimage.binary_dilation(invalide)
prof = np.where(invalide, 0, reconstruction(np.where(exut, z_rec, z_rec.max()), z_rec, method='erosion') - z_rec)
etiq, n_dep = ndimage.label(prof >= PROF_MIN_M)
aires = ndimage.sum(np.ones(sh_fab), etiq, index=np.arange(1, n_dep + 1)) * pas_fab ** 2 / 1e6
garder = np.isin(etiq, np.arange(1, n_dep + 1)[aires >= AIRE_MIN_KM2])
prof = np.where(garder, prof, 0).astype('float32')
etiq_g, n_g = ndimage.label(garder)
print(f'Dépressions fermées (≥ {PROF_MIN_M} m, ≥ {AIRE_MIN_KM2} km²) : {n_g} ; '
      f'{garder.sum() * pas_fab ** 2 / 1e6:.1f} km²')

grid = Grid.from_raster(F_FAB_NET, nodata=NODATA_DEM)
dem = grid.read_raster(F_FAB_NET, nodata=NODATA_DEM)
acc = grid.accumulation(grid.flowdir(grid.resolve_flats(grid.fill_pits(dem))))
aire_km2 = np.where(invalide, 0, np.asarray(acc, dtype='float32') * pas_fab ** 2 / 1e6).astype('float32')

def vers_eco(arr):
    out = np.zeros((H, W), 'float32')
    reproject(arr, out, src_transform=tr_fab, src_crs=crs_fab, dst_transform=T_ECO, dst_crs=CRS_M,
              resampling=Resampling.max)
    return out
AIRE_ECO, PROF_ECO = vers_eco(aire_km2), vers_eco(prof)
Z_ECO = np.full((H, W), np.nan, 'float32')
reproject(z_net, Z_ECO, src_transform=tr_fab, src_crs=crs_fab, src_nodata=NODATA_DEM,
          dst_transform=T_ECO, dst_crs=CRS_M, dst_nodata=np.nan, resampling=Resampling.average)
for nom, arr in [('surface_amont_km2', aire_km2), ('profondeur_depressions_m', prof)]:
    ecrire_tif(os.path.join(SORTIES, f'FABDEM_{nom}_30m.tif'), [arr], [nom], tr_fab, crs_fab)

ANOM_ND = anom_periode('nuit', [11, 12])
dec = MASQUE_ZET & OUVERT & np.isfinite(ANOM_ND)
lig = [{'classe': f'surface amont {lab}', 'pixels': int((dec & (AIRE_ECO >= a) & (AIRE_ECO < b)).sum()),
        'anomalie nuit nov.–déc. (°C)': np.nanmean(ANOM_ND[dec & (AIRE_ECO >= a) & (AIRE_ECO < b)])}
       for a, b, lab in [(0, 0.01, '< 0,01 km²'), (0.01, 0.1, '0,01–0,1'), (0.1, 1, '0,1–1'),
                         (1, 10, '1–10'), (10, np.inf, '> 10 km²')]]
dep_ = PROF_ECO >= PROF_MIN_M
lig += [{'classe': lab, 'pixels': int(m_.sum()), 'anomalie nuit nov.–déc. (°C)': np.nanmean(ANOM_ND[m_])}
        for lab, m_ in [('dans une dépression fermée', dec & dep_), ('hors dépression fermée', dec & ~dep_)]]
T = pd.DataFrame(lig)
print('\nPixels découverts de la ZET :'); print(T.round(2).to_string(index=False))
ecrire_csv(T.round(3), 'anomalie_surface_amont_depressions.csv')

xs_f = tr_fab.c + pas_fab * (np.arange(sh_fab[1]) + 0.5); ys_f = tr_fab.f - pas_fab * (np.arange(sh_fab[0]) + 0.5)
ext_f = (tr_fab.c, tr_fab.c + pas_fab * sh_fab[1], tr_fab.f - pas_fab * sh_fab[0], tr_fab.f)
violet = ListedColormap(['#40004b']); violet.set_bad(alpha=0)
v = float(np.nanpercentile(np.abs(ANOM_ND), 98))
cmap = plt.get_cmap('RdBu_r').copy(); cmap.set_bad('#cfcfcf')
fig, ax = plt.subplots(figsize=(12, 11))
ax.imshow(ANOM_ND, cmap=cmap, vmin=-v, vmax=v, extent=EXT)
ax.imshow(hors_zet, cmap=blanc, extent=EXT)
ax.imshow(np.where(aire_km2 >= SEUIL_RESEAU_KM2, 1.0, np.nan), cmap=violet, extent=ext_f, interpolation='nearest')
if garder.any():
    ax.contour(xs_f, ys_f, garder.astype('float32'), levels=[0.5], colors='#e66101', linewidths=0.9)
tracer_contour(ax, zet_m, color='k', lw=1)
ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_axis_off()
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(-v, v)), ax=ax, shrink=0.6,
             label='Anomalie de LST de nuit, nov.–déc. (°C)')
ax.set_title(f'Écoulement potentiel (violet : surface amont ≥ {SEUIL_RESEAU_KM2} km²) et dépressions fermées (orange)',
             fontsize=10)
sauver_figure(fig, 'ecoulement_depressions_nov_dec'); plt.show()

In [ ]:
# ============================================================
# 14. DÉPRESSIONS FERMÉES : POLYGONES, POINT DU FOND, ATTRIBUTS
# ============================================================
lab = etiq_g.astype('int32'); ids = np.arange(1, n_g + 1)
morceaux = {}
for g, val in shapes(lab, mask=lab > 0, transform=tr_fab):
    morceaux.setdefault(int(val), []).append(shape(g))
poly = [unary_union(morceaux[i]) for i in ids]                # poly[id_dep − 1]

pos = ndimage.maximum_position(prof, lab, ids)
fond_xy = [(tr_fab.c + (c + 0.5) * pas_fab, tr_fab.f - (r + 0.5) * pas_fab) for r, c in pos]
lab_eco = np.zeros((H, W), 'int32')
reproject(lab, lab_eco, src_transform=tr_fab, src_crs=crs_fab, dst_transform=T_ECO, dst_crs=CRS_M,
          resampling=Resampling.nearest)

def moy_dep(arr):
    v = np.isfinite(arr) & (lab_eco > 0)
    s, c = ndimage.sum(np.where(v, arr, 0), lab_eco, ids), ndimage.sum(v, lab_eco, ids)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan), c.astype(int)

to_ll = Transformer.from_crs(crs_fab, 'EPSG:4326', always_xy=True)
lon_f, lat_f = zip(*[to_ll.transform(x, y) for x, y in fond_xy])
df = pd.DataFrame({'id_dep': ids,
                   'surface_km2': ndimage.sum(np.ones_like(lab), lab, ids) * pas_fab ** 2 / 1e6,
                   'prof_max_m': ndimage.maximum(prof, lab, ids),
                   'alt_fond_m': ndimage.minimum(np.where(lab > 0, z, np.inf), lab, ids),
                   'amont_max_km2': ndimage.maximum(aire_km2, lab, ids),
                   'part_ZET': [p.intersection(zet_m).area / p.area for p in poly],
                   'part_foret': moy_dep(FRAC_FORET)[0],
                   'lon_fond': lon_f, 'lat_fond': lat_f})
df['n_pix_eco'] = moy_dep(ANOM_ND)[1]
df['anom_nuit_ND'] = moy_dep(ANOM_ND)[0]
df['anom_jour_ND'] = moy_dep(anom_periode('jour', [11, 12]))[0]
for m, c in [('nuit', 'n'), ('jour', 'j')]:
    for mm in range(1, 13):
        df[f'anom_{c}_m{mm:02d}'] = moy_dep(clim[(m, mm)][0])[0]
if os.path.exists(F['microrefuges']):                          # microrefuge MODIS le plus proche (facultatif)
    mr = pd.read_csv(F['microrefuges'])
    if {'x_merchich', 'y_merchich', 'maille'} <= set(mr.columns):
        pts_mr = [Point(x, y) for x, y in zip(mr['x_merchich'], mr['y_merchich'])]
        dmr = [[p.distance(q) for q in pts_mr] for p in poly]
        df['mr_proche'] = [mr['maille'].iloc[int(np.argmin(d))] for d in dmr]
        df['dist_mr_m'] = [min(d) for d in dmr]
df = df.round({'surface_km2': 4, 'prof_max_m': 1, 'alt_fond_m': 0, 'amont_max_km2': 3, 'part_ZET': 2,
               'part_foret': 2, 'lon_fond': 5, 'lat_fond': 5, 'dist_mr_m': 0}).round(
               {c: 3 for c in df.columns if c.startswith('anom')})

gpoly = gpd.GeoDataFrame(df, geometry=poly, crs=EPSG_VECT)
gpts = gpd.GeoDataFrame(df.copy(), geometry=[Point(xy) for xy in fond_xy], crs=EPSG_VECT)
print('GeoPackage :', ecrire_vecteurs('depressions_fermees', {'depressions_polygones': gpoly,
                                                              'depressions_points_fond': gpts}))
print('CSV :', ecrire_csv(df, 'depressions_fermees_attributs.csv'))
print(f'{len(df)} dépressions ; {(df["part_ZET"] >= 0.5).sum()} dans la ZET')

In [ ]:
# ============================================================
# 15. ÉCART FOND − BORDS PAR PASSAGE ET MODÈLE DE DÉVELOPPEMENT NOCTURNE
# ============================================================
# Réf. : indicateurs CAP — Noad & Bonnaventure (2022, Arctic Science ; repris par Pastore et al. 2024) ;
# Jönsson & Eklundh (2002, IEEE TGRS ; 2004, Comput. Geosci.) — modèle harmonique ;
# Efron & Tibshirani (1986, Stat. Sci. ; 1993) — bootstrap (IC 95 %) ;
# Clements et al. (2003) — développement nocturne.
# Fond : pixels découverts ≥ 50 % de la profondeur maximale (≥ N_FOND_MIN valides).
# Bords : anneau de 100 m à 2 × rayon équivalent (borné à 300–1 000 m), découvert, hors autres dépressions.
# Modèle (nuits solaires) : ΔT = a + b·p + c·cos(2πj/365,25) + d·sin(2πj/365,25), p = part de nuit écoulée ;
# ΔT prédit en début (0,2) et fin (0,9) de nuit pour chaque mois ; IC 95 % par rééchantillonnage des nuits.
hors_dep = lab_eco == 0
ZONES = {}
for i, p in enumerate(poly, start=1):
    dep = lab_eco == i
    fond = dep & (PROF_ECO >= 0.5 * float(df.at[i - 1, 'prof_max_m'])) & OUVERT
    if fond.sum() < N_FOND_MIN:
        fond = dep & OUVERT
    if fond.sum() < N_FOND_MIN:
        continue
    r_out = float(np.clip(2 * np.sqrt(p.area / np.pi), R_ANNEAU_MIN, R_ANNEAU_MAX))
    bords = ras(p.buffer(r_out)) & ~ras(p.buffer(R_TAMPON)) & OUVERT & hors_dep
    if bords.sum() >= N_BORD_MIN:
        ZONES[i] = (np.flatnonzero(fond), np.flatnonzero(bords))
print(f'Dépressions avec fond et bords découverts suffisants : {len(ZONES)} / {len(poly)}')

rec = []
for f, g in META.groupby('fichier'):
    with nc.Dataset(f) as ds:
        for _, r in g.iterrows():
            k = int(r['k'])
            L = np.ma.filled(ds['LST'][k, :, :], np.nan).astype('f4').ravel()
            S = np.ma.filled(ds['neige'][k, :, :], 255).ravel()
            L[S == 1] = np.nan
            for i, (jf, jb) in ZONES.items():
                vf, vb = L[jf], L[jb]
                nf, nb = int(np.isfinite(vf).sum()), int(np.isfinite(vb).sum())
                if nf >= N_FOND_MIN and nb >= N_BORD_MIN:
                    tf, tb = float(np.nanmean(vf)), float(np.nanmean(vb))
                    rec.append((i, r['utc'], int(r['nuit_solaire']), r['part_nuit_ecoulee'], r['h_depuis_coucher'],
                                r['part_jour_ecoulee'], tf, tb, tf - tb, nf, nb, float((S[jf] == 1).mean())))
R = pd.DataFrame(rec, columns=['id_dep', 'utc', 'nuit_solaire', 'part_nuit', 'h_depuis_coucher', 'part_jour',
                               'T_fond', 'T_bords', 'dT', 'n_fond', 'n_bords', 'neige_fond'])
ecrire_csv(R, 'cuvettes_fond_bords_par_passage.csv.gz', compression='gzip')
Rn = R[R['nuit_solaire'] == 1].dropna(subset=['part_nuit', 'dT']).copy()
Rn['annee'] = Rn['utc'].dt.year
print(f'Couples (dépression, passage) : {len(R)} ; de nuit solaire : {len(Rn)}')

DOY_MOIS = np.array([pd.Timestamp(2023, m, 15).dayofyear for m in range(1, 13)])
def X_mat(part, doy):
    w = 2 * np.pi * np.asarray(doy) / 365.25
    return np.column_stack([np.ones(len(w)), part, np.cos(w), np.sin(w)])
def predire(beta, part):
    return X_mat(np.full(12, part), DOY_MOIS) @ beta
def ajuster(d):
    """Coefficients, prédictions et IC 95 % (rééchantillonnage des nuits) pour une dépression."""
    X, y = X_mat(d['part_nuit'].to_numpy(), d['utc'].dt.dayofyear.to_numpy()), d['dT'].to_numpy()
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    bs = np.array([np.linalg.lstsq(X[j], y[j], rcond=None)[0]
                   for j in (rng.integers(0, len(y), len(y)) for _ in range(N_BOOT))])
    nd = np.array([predire(b_, PART_FIN)[[10, 11]].mean() for b_ in bs])
    return beta, bs, nd

lignes, courbes = [], []
for i, d in Rn.groupby('id_dep'):
    if len(d) < N_NUITS_MODELE:
        continue
    beta, bs, nd = ajuster(d)
    fin, deb = predire(beta, PART_FIN), predire(beta, PART_DEBUT)
    tard = d[d['part_nuit'] > 0.6]
    lignes.append({'id_dep': i, 'n_nuits': len(d), 'n_fin_nuit': len(tard),
                   'a': beta[0], 'b_developpement': beta[1], 'c_cos': beta[2], 'd_sin': beta[3],
                   'b_IC_bas': np.percentile(bs[:, 1], 2.5), 'b_IC_haut': np.percentile(bs[:, 1], 97.5),
                   'dT_fin_ND': fin[[10, 11]].mean(), 'dT_fin_ND_IC_bas': np.percentile(nd, 2.5),
                   'dT_fin_ND_IC_haut': np.percentile(nd, 97.5), 'dT_debut_ND': deb[[10, 11]].mean(),
                   'dT_fin_JJA': fin[[5, 6, 7]].mean(), 'mois_min_fin': NOMS_MOIS[int(np.argmin(fin))],
                   'part_inv_fin_obs': (tard['dT'] < 0).mean() if len(tard) else np.nan})
    courbes.append(pd.DataFrame({'id_dep': i, 'mois': range(1, 13), 'fin': fin, 'debut': deb}))
MOD = pd.DataFrame(lignes)
MOD['classe'] = np.select([MOD['dT_fin_ND_IC_haut'] < 0, MOD['dT_fin_ND'] < 0],
                          ['piège robuste', 'inversion faible'], 'pas d\'inversion')
C = pd.concat(courbes, ignore_index=True)
ecrire_csv(MOD.round(4), 'cuvettes_modele_nocturne.csv')

print(f'\nDépressions modélisées (≥ {N_NUITS_MODELE} nuits) : {len(MOD)} ; classes : {MOD["classe"].value_counts().to_dict()}')
print(f'b (coucher -> lever) : médiane {MOD["b_developpement"].median():+.2f} °C ; IC < 0 pour {(MOD["b_IC_haut"] < 0).mean():.0%}')
print(f'ΔT fin de nuit nov.–déc. : médiane {MOD["dT_fin_ND"].median():+.2f} °C ; < 0 pour {(MOD["dT_fin_ND"] < 0).mean():.0%} ; '
      f'IC < 0 pour {(MOD["dT_fin_ND_IC_haut"] < 0).mean():.0%} ; fin de nuit juin–août : '
      f'médiane {MOD["dT_fin_JJA"].median():+.2f} °C')
print('Mois du minimum prédit (fin de nuit) :', MOD['mois_min_fin'].value_counts().reindex(NOMS_MOIS, fill_value=0).to_dict())

gmod = gpoly[['id_dep', 'surface_km2', 'prof_max_m', 'alt_fond_m', 'lon_fond', 'lat_fond', 'geometry']] \
       .merge(MOD.round(4), on='id_dep', how='inner')
print('GeoPackage :', ecrire_vecteurs('depressions_modele_nocturne', {'depressions_modele_nocturne': gmod}))

pn_ = Rn.groupby('utc').agg(dT=('dT', 'mean'), n_dep=('id_dep', 'size'), part=('part_nuit', 'first'))
pn_ = pn_[pn_['n_dep'] >= 20].reset_index()
pn_['mois'] = pn_['utc'].dt.month
pn_['classe'] = np.where(pn_['part'] < 0.4, 'début', np.where(pn_['part'] > 0.6, 'fin', 'milieu'))
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
for classe, coul in [('début', '#fd8d3c'), ('fin', '#08306b')]:
    s = pn_[pn_['classe'] == classe]
    gm = s.groupby('mois')['dT'].agg(['mean', 'sem', 'size']).reindex(range(1, 13))
    for m, r in gm.iterrows():
        if np.isfinite(r['mean']):
            ax[0].errorbar(m, r['mean'], yerr=1.96 * (r['sem'] if r['size'] > 1 else 0), fmt='o', color=coul,
                           mfc=coul if r['size'] >= 5 else 'white', capsize=3)
    ax[0].plot(range(1, 13), [s[s['mois'].isin([(m - 2) % 12 + 1, m, m % 12 + 1])]['dT'].mean() for m in range(1, 13)],
               '-', color=coul, lw=1.8, label=f'{classe} de nuit (3 mois glissants)')
for col, coul, lab_ in [('debut', '#fd8d3c', f'début de nuit ({PART_DEBUT})'), ('fin', '#08306b', f'fin de nuit ({PART_FIN})')]:
    q = C.groupby('mois')[col].quantile([0.25, 0.5, 0.75]).unstack()
    ax[1].fill_between(q.index, q[0.25], q[0.75], color=coul, alpha=0.15)
    ax[1].plot(q.index, q[0.5], '-o', color=coul, label=f'modèle, {lab_} : médiane et quartiles')
for a_ in ax:
    a_.axhline(0, color='k', lw=0.8); a_.set_xticks(range(1, 13)); a_.set_xticklabels(NOMS_MOIS, rotation=45)
    a_.legend(fontsize=9)
ax[0].set_ylabel('Écart fond − bords (°C)')
ax[0].set_title('Observé (IC 95 % ; symbole vide si < 5 nuits)', fontsize=10)
ax[1].set_title(f'Modèle ({len(MOD)} dépressions), mois comparés à heure égale', fontsize=10)
plt.tight_layout(); sauver_figure(fig, 'cuvettes_saisonnalite_heure_egale'); plt.show()

In [ ]:
# ============================================================
# 16. DÉTERMINANTS DES PIÈGES À FROID (forme, relief, alimentation, position, couvert)
# ============================================================
# Réf. : Mann & Whitney (1947, Ann. Math. Stat.) — U ; Kerby (2014, Compr. Psychol.) —
# taille d'effet r = 2U/(n1·n2) − 1 ; Spearman (1904) — ρ ; Hanley & McNeil (1982, Radiology)
# — AUC ; Pedregosa et al. (2011, JMLR) — scikit-learn (logistique, CV stratifiée 5 plis).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

gy, gx = np.gradient(np.where(invalide, np.nan, z_net.astype('f8')), pas_fab)
PENTE_ECO = np.full((H, W), np.nan, 'float32')
reproject(np.nan_to_num(np.degrees(np.arctan(np.hypot(gx, gy))), nan=-9999).astype('float32'), PENTE_ECO,
          src_transform=tr_fab, src_crs=crs_fab, src_nodata=-9999, dst_transform=T_ECO, dst_crs=CRS_M,
          dst_nodata=np.nan, resampling=Resampling.average)
zf, pf, ff = Z_ECO.ravel(), PENTE_ECO.ravel(), FRAC_FORET.ravel()
desc = []
for i in MOD['id_dep']:
    jf, jb = ZONES[i]; p = poly[i - 1]; a = df.iloc[i - 1]
    entour = np.flatnonzero(ras(p.buffer(1000)) & ~ras(p))
    desc.append({'id_dep': i, 'n_pix_fond': len(jf), 'surface_km2': a['surface_km2'], 'prof_max_m': a['prof_max_m'],
                 'prof_sur_diametre': a['prof_max_m'] / (2 * np.sqrt(p.area / np.pi)),
                 'denivele_bords_m': np.nanmean(zf[jb]) - np.nanmean(zf[jf]),
                 'relief_max_bords_m': np.nanmax(zf[jb]) - np.nanmean(zf[jf]),
                 'pente_bords_deg': np.nanmean(pf[jb]), 'amont_km2': a['amont_max_km2'], 'alt_fond_m': a['alt_fond_m'],
                 'foret_1km': np.nanmean(ff[entour]), 'lon': a['lon_fond'], 'lat': a['lat_fond']})
D = pd.DataFrame(desc).merge(MOD[['id_dep', 'n_nuits', 'dT_fin_ND', 'dT_fin_ND_IC_haut', 'b_developpement', 'classe']],
                             on='id_dep')
VARS = ['surface_km2', 'prof_max_m', 'prof_sur_diametre', 'denivele_bords_m', 'relief_max_bords_m',
        'pente_bords_deg', 'amont_km2', 'alt_fond_m', 'foret_1km', 'lon', 'lat']
VARS_LOG = ['surface_km2', 'prof_max_m', 'prof_sur_diametre', 'denivele_bords_m', 'pente_bords_deg',
            'amont_km2', 'alt_fond_m', 'foret_1km']

def comparer(tab):
    """Taille d'effet rang-bisériel (pièges robustes contre autres), p de Mann-Whitney, ρ de Spearman avec ΔT."""
    rob, aut = tab[tab['classe'] == 'piège robuste'], tab[tab['classe'] != 'piège robuste']
    out = []
    for v in VARS:
        x, y = rob[v].dropna(), aut[v].dropna()
        u, p_ = mannwhitneyu(x, y) if len(x) >= 5 and len(y) >= 5 else (np.nan, np.nan)
        out.append({'descripteur': v, 'médiane pièges': x.median(), 'médiane autres': y.median(),
                    'effet': 2 * u / (len(x) * len(y)) - 1 if np.isfinite(u) else np.nan, 'p': p_,
                    'rho avec ΔT': spearmanr(tab[v], tab['dT_fin_ND'], nan_policy='omit')[0]})
    return pd.DataFrame(out)

def logistique(tab):
    d = tab.dropna(subset=VARS_LOG); X = d[VARS_LOG].copy()
    X[['surface_km2', 'amont_km2']] = np.log10(X[['surface_km2', 'amont_km2']])
    y = (d['classe'] == 'piège robuste').astype(int)
    k = min(5, int(y.sum()), int(len(y) - y.sum()))
    if k < 2:
        return None, None
    mod = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    auc = cross_val_score(mod, X, y, cv=StratifiedKFold(k, shuffle=True, random_state=SEED), scoring='roc_auc')
    mod.fit(X, y)
    return auc, pd.Series(mod[-1].coef_[0], index=VARS_LOG).sort_values(key=np.abs, ascending=False)

T1 = comparer(D)
print('Effectifs :', D['classe'].value_counts().to_dict())
print(T1.round(3).sort_values('effet', key=np.abs, ascending=False).to_string(index=False))
auc, coefs = logistique(D)
if auc is not None:
    print(f'\nRégression logistique standardisée : AUC = {auc.mean():.2f} ± {auc.std():.2f}'); print(coefs.round(2).to_string())
ecrire_csv(D.round(4), 'cuvettes_descripteurs.csv'); ecrire_csv(T1.round(4), 'cuvettes_determinants.csv')

gcl = gpoly[['id_dep', 'geometry']].merge(D.round(4), on='id_dep', how='inner')
ecrire_vecteurs('depressions_classes_pieges', {'depressions_classes': gcl})
ordre = ['piège robuste', 'inversion faible', 'pas d\'inversion']
coul = {'piège robuste': '#08306b', 'inversion faible': '#6baed6', 'pas d\'inversion': '#bdbdbd'}
fig, ax = plt.subplots(figsize=(10, 10))
tracer_contour(ax, zet_m, color='k', lw=1)
for c in ordre[::-1]:
    s = gcl[gcl['classe'] == c]; pt = s.geometry.representative_point()
    ax.scatter(pt.x, pt.y, s=10 + 300 * np.sqrt(s['surface_km2']), c=coul[c], edgecolor='k', linewidth=0.3,
               label=f'{c} ({len(s)})')
ax.set_aspect('equal'); ax.set_axis_off(); ax.legend(loc='lower left')
ax.set_title('Dépressions fermées découvertes : inversion de fin de nuit (nov.–déc.)', fontsize=11)
sauver_figure(fig, 'carte_classes_pieges_froid'); plt.show()

In [ ]:
# ============================================================
# 17. ROBUSTESSE À LA RÉSOLUTION : DÉPRESSIONS À FOND BIEN RÉSOLU (≥ N_FOND_RESOLU pixels)
# ============================================================
# Réf. : Efron & Tibshirani (1986, Stat. Sci.) — bootstrap (rééchantillonnage des nuits).
Ds = D[D['n_pix_fond'] >= N_FOND_RESOLU].copy()
print(f'Fonds ≥ {N_FOND_RESOLU} pixels : {len(Ds)} / {len(D)} ; classes : {Ds["classe"].value_counts().to_dict()}')
T2 = comparer(Ds)
comp = T1[['descripteur', 'effet', 'rho avec ΔT']].merge(T2[['descripteur', 'effet', 'p', 'rho avec ΔT']],
                                                         on='descripteur', suffixes=(' (toutes)', ' (fond résolu)'))
print(comp.round(3).sort_values('effet (toutes)', key=np.abs, ascending=False).to_string(index=False))
ecrire_csv(comp.round(4), 'cuvettes_determinants_fond_resolu.csv')
if Ds['classe'].eq('piège robuste').sum() >= 10 and Ds['classe'].ne('piège robuste').sum() >= 10:
    auc2, co2 = logistique(Ds)
    if auc2 is not None:
        print(f'\nLogistique (fond résolu) : AUC = {auc2.mean():.2f} ± {auc2.std():.2f}'); print(co2.round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
for tab, lab_, c in [(D[D['n_pix_fond'] < N_FOND_RESOLU], f'fond < {N_FOND_RESOLU} pixels', '#bdbdbd'),
                     (Ds, f'fond ≥ {N_FOND_RESOLU} pixels', '#08306b')]:
    ax.scatter(tab['surface_km2'], tab['dT_fin_ND'], s=22, c=c, edgecolor='k', linewidth=0.2, label=f'{lab_} (n = {len(tab)})')
ax.set_xscale('log'); ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Surface de la dépression (km², échelle log)')
ax.set_ylabel('Écart fond − bords prédit, fin de nuit nov.–déc. (°C)'); ax.legend()
sauver_figure(fig, 'test_taille_fond_resolu'); plt.show()

In [ ]:
# ============================================================
# 18. STABILITÉ ENTRE PÉRIODES (2021–2023 contre 2024–2026)
# ============================================================
# Réf. : Cohen (1960, Educ. Psychol. Meas.) — kappa ; Wilcoxon (1945, Biometrics Bull.) —
# rangs signés ; Barnett et al. (2005, Int. J. Epidemiol.) — régression vers la moyenne ;
# Good (2005, Springer) — tests de permutation.
# Même modèle, ajusté sur chaque période ; accord des classes (kappa), corrélation des écarts, régression vers
# la moyenne (sélection dans un sens puis dans l'autre), variations standardisées z et facteurs associés.
from sklearn.metrics import cohen_kappa_score
p1, p2 = list(PERIODES_STAB)
lig = []
for i, d_all in Rn.groupby('id_dep'):
    l_ = {'id_dep': i}
    for nom, (a0, a1) in PERIODES_STAB.items():
        d = d_all[d_all['annee'].between(a0, a1)]
        l_[f'n_{nom}'] = len(d)
        if len(d) >= N_NUITS_PERIODE:
            beta, bs, nd = ajuster(d)
            est, (bas, haut) = predire(beta, PART_FIN)[[10, 11]].mean(), np.percentile(nd, [2.5, 97.5])
            l_.update({f'dT_{nom}': est, f'bas_{nom}': bas, f'haut_{nom}': haut,
                       f'classe_{nom}': 'piège robuste' if haut < 0 else ('inversion faible' if est < 0 else 'pas d\'inversion')})
    lig.append(l_)
ST = pd.DataFrame(lig).dropna(subset=[f'classe_{p1}', f'classe_{p2}'])
kappa = cohen_kappa_score(ST[f'classe_{p1}'], ST[f'classe_{p2}'], labels=ordre)
rho = spearmanr(ST[f'dT_{p1}'], ST[f'dT_{p2}'])[0]
r1, r2 = ST[f'classe_{p1}'] == 'piège robuste', ST[f'classe_{p2}'] == 'piège robuste'
print(f'{len(ST)} dépressions ; même classe : {(ST[f"classe_{p1}"] == ST[f"classe_{p2}"]).mean():.0%} ; '
      f'kappa = {kappa:.2f} ; ρ des écarts = {rho:.2f}')
print(f'Pièges robustes en {p1} toujours en inversion en {p2} : {(r1 & (ST[f"dT_{p2}"] < 0)).sum()} / {r1.sum()}')
print(pd.crosstab(ST[f'classe_{p1}'], ST[f'classe_{p2}']).reindex(index=ordre, columns=ordre, fill_value=0).to_string())

def variation(sel, de, vers, nom):
    a_, b_ = ST.loc[sel, f'dT_{de}'], ST.loc[sel, f'dT_{vers}']
    return {'groupe': nom, 'n': int(sel.sum()), 'écart (sélection)': a_.mean(), 'écart (autre période)': b_.mean(),
            'variation (°C)': (b_ - a_).mean(), 'p Wilcoxon': wilcoxon(a_, b_).pvalue if sel.sum() >= 6 else np.nan}
RM = pd.DataFrame([variation(pd.Series(True, index=ST.index), p1, p2, 'toutes les dépressions'),
                   variation(r1, p1, p2, f'robustes en {p1}'), variation(r2, p2, p1, f'robustes en {p2} (à rebours)')])
print('\nRégression vers la moyenne (variation > 0 = inversion plus faible dans l\'autre période) :')
print(RM.round(3).to_string(index=False))
print(f'Pentes : {np.polyfit(ST[f"dT_{p1}"], ST[f"dT_{p2}"], 1)[0]:.2f} et '
      f'{np.polyfit(ST[f"dT_{p2}"], ST[f"dT_{p1}"], 1)[0]:.2f} (< 1 dans les deux sens : régression vers la moyenne)')

se = lambda n_: (ST[f'haut_{n_}'] - ST[f'bas_{n_}']) / 3.92
ST['delta'] = ST[f'dT_{p2}'] - ST[f'dT_{p1}']
ST['z'] = ST['delta'] / np.sqrt(se(p1) ** 2 + se(p2) ** 2)
ST['changement_net'] = (ST[f'haut_{p1}'] < ST[f'bas_{p2}']) | (ST[f'haut_{p2}'] < ST[f'bas_{p1}'])
print(f'\nz : écart-type {ST["z"].std():.2f} (hasard : 1) ; |z| > 1,96 : {(ST["z"].abs() > 1.96).mean():.1%} '
      f'(attendu 5 %) ; changements nets (IC disjoints) : {int(ST["changement_net"].sum())}')
pts = gpoly.set_index('id_dep').loc[ST['id_dep']].geometry.representative_point()
xy = np.column_stack([pts.x, pts.y]); _, vois = cKDTree(xy).query(xy, k=min(9, len(xy)))
dz_ = ST['delta'].to_numpy()
corr_v = lambda v_: np.corrcoef(v_, v_[vois[:, 1:]].mean(axis=1))[0, 1]
obs = corr_v(dz_); perm = np.array([corr_v(rng.permutation(dz_)) for _ in range(999)])
print(f'Ressemblance avec les 8 voisines : r = {obs:.2f} ; p (permutation) = {(perm >= obs).mean():.3f}')
ST = ST.merge(D[['id_dep'] + VARS], on='id_dep', how='left')
print('Corrélation de Spearman des descripteurs avec la variation :',
      {v: round(float(spearmanr(ST[v], ST['delta'], nan_policy='omit')[0]), 2) for v in VARS})
ecrire_csv(ST.round(4), 'stabilite_periodes.csv'); ecrire_csv(RM.round(4), 'stabilite_regression_moyenne.csv')
ecrire_vecteurs('depressions_stabilite', {'stabilite': gpoly[['id_dep', 'geometry']].merge(ST.round(4), on='id_dep')})

fig, axs = plt.subplots(1, 2, figsize=(15, 6))
ax = axs[0]
ax.errorbar(ST[f'dT_{p1}'], ST[f'dT_{p2}'], xerr=[ST[f'dT_{p1}'] - ST[f'bas_{p1}'], ST[f'haut_{p1}'] - ST[f'dT_{p1}']],
            yerr=[ST[f'dT_{p2}'] - ST[f'bas_{p2}'], ST[f'haut_{p2}'] - ST[f'dT_{p2}']], fmt='o', ms=4,
            color='#2171b5', elinewidth=0.4, alpha=0.8)
lim = [min(ST[f'dT_{p1}'].min(), ST[f'dT_{p2}'].min()) - 0.3, max(ST[f'dT_{p1}'].max(), ST[f'dT_{p2}'].max()) + 0.3]
ax.plot(lim, lim, 'k--', lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(f'ΔT fin de nuit nov.–déc., {p1} (°C)'); ax.set_ylabel(f'Idem, {p2} (°C)')
ax.set_title(f'Stabilité entre périodes (ρ = {rho:.2f})', fontsize=11)
ax = axs[1]
ax.hist(ST['z'], bins=30, density=True, color='#9ecae1', edgecolor='white', label='z observés')
zz_ = np.linspace(-4, 4, 200); ax.plot(zz_, norm.pdf(zz_), 'k-', label='hasard N(0, 1)'); ax.legend()
ax.set_xlabel('Variation standardisée z (> 0 : inversion affaiblie)')
ax.set_title('Les variations dépassent-elles le hasard ?', fontsize=11)
plt.tight_layout(); sauver_figure(fig, 'stabilite_periodes'); plt.show()

## D. Exemples de cuvettes (analyse complémentaire, figures de publication)

Pour une ou plusieurs dépressions : carte de la LST de fin de nuit d'hiver (pixels de 70 m, fond, tampon, anneau des
bords), affichage combiné sur image satellite avec profil fond → bords, série nuit par nuit et cycle saisonnier en
début et fin de nuit. Exports : figures (PNG, PDF), GeoTIFF de la composite, CSV du profil et de la série.

In [ ]:
# ============================================================
# 19. EXEMPLES DE CUVETTES : CARTES DE LST DE FIN DE NUIT ET PROFIL FOND -> BORDS
# ============================================================
# Réf. : Fond : Esri World Imagery (Esri Master License Agreement) ; crédits affichés sur chaque
# carte, conformément aux règles d'Esri pour les publications ; Hunter (2007) — matplotlib.
EXEMPLES_DEP = None          # None : la dépression la plus forte du modèle ; sinon une liste, par ex. [362, 310]
MOIS_EXEMPLE = [11, 12, 1]   # fin de nuit d'hiver
PART_MIN_EX = 0.6            # passages au-delà de 60 % de la nuit écoulée
N_MIN_PIX_EX = 3             # nuits minimales pour afficher un pixel
ALPHA_LST, N_RAYONS = 0.55, 8

if FIGURES_EXEMPLES:
    try:
        import contextily as ctx
    except ImportError:
        ctx = None
    EX = ([int(MOD.sort_values('dT_fin_ND').iloc[0]['id_dep'])] if EXEMPLES_DEP is None
          else [i for i in EXEMPLES_DEP if i in set(MOD['id_dep'])])
    fr = lambda v, d=1: f'{v:.{d}f}'.replace('.', ',').replace('-', '−')
    cmap_ex = LinearSegmentedColormap.from_list('froid_chaud', ['#08306b', '#2171b5', '#6baed6', '#c6dbef',
                                                                '#f0f0f0', '#fdd0a2', '#fd8d3c'])
    sel_ex = META[(META['nuit_solaire'] == 1) & (META['part_nuit_ecoulee'] > PART_MIN_EX)
                  & META['mois'].isin(MOIS_EXEMPLE)]

    def echant(mk, n_max=40):
        """Sous-échantillonne régulièrement un masque trop dense pour rester lisible."""
        if mk.sum() <= n_max:
            return mk
        s_ = int(np.ceil(np.sqrt(mk.sum() / n_max))); g_ = np.zeros_like(mk); g_[::s_, ::s_] = True
        return mk & g_

    def masque_plat(idx):
        m_ = np.zeros(H * W, bool); m_[idx] = True; return m_.reshape(H, W)

    def contour_pixels(ax, masque, ext, **kw):
        h_, w_ = masque.shape
        for i, j in zip(*np.where(masque)):
            xa, yb = ext[0] + j * PAS_ECO, ext[3] - i * PAS_ECO
            for cond, seg in [(i == 0 or not masque[i - 1, j], ([xa, xa + PAS_ECO], [yb, yb])),
                              (i == h_ - 1 or not masque[i + 1, j], ([xa, xa + PAS_ECO], [yb - PAS_ECO] * 2)),
                              (j == 0 or not masque[i, j - 1], ([xa, xa], [yb - PAS_ECO, yb])),
                              (j == w_ - 1 or not masque[i, j + 1], ([xa + PAS_ECO] * 2, [yb - PAS_ECO, yb]))]:
                if cond:
                    ax.plot(*seg, **kw)

    for i in EX:
        p, m = poly[i - 1], MOD[MOD['id_dep'] == i].iloc[0]
        r_out = float(np.clip(2 * np.sqrt(p.area / np.pi), R_ANNEAU_MIN, R_ANNEAU_MAX))
        marge = r_out + 2 * PAS_ECO
        bx0, by0, bx1, by1 = p.bounds
        cols = np.where((xs_e >= bx0 - marge) & (xs_e <= bx1 + marge))[0]
        rows = np.where((ys_e >= by0 - marge) & (ys_e <= by1 + marge))[0]
        r0, r1, c0, c1 = rows.min(), rows.max() + 1, cols.min(), cols.max() + 1
        sl, h, w = (slice(r0, r1), slice(c0, c1)), r1 - r0, c1 - c0
        ext = (x0 + c0 * PAS_ECO, x0 + c1 * PAS_ECO, y1 - r1 * PAS_ECO, y1 - r0 * PAS_ECO)
        fond, anneau = masque_plat(ZONES[i][0])[sl], masque_plat(ZONES[i][1])[sl]
        dans = (lab_eco == i)[sl]
        exclus = (ras(p.buffer(r_out)) & ~ras(p.buffer(R_TAMPON)))[sl] & ~anneau & ~dans

        somme, n = np.zeros((h, w)), np.zeros((h, w))
        for f, g in sel_ex.groupby('fichier'):
            with nc.Dataset(f) as ds:
                for k in g['k']:
                    L = np.ma.filled(ds['LST'][int(k), r0:r1, c0:c1], np.nan).astype('f8')
                    L[np.ma.filled(ds['neige'][int(k), r0:r1, c0:c1], 255) == 1] = np.nan
                    v = np.isfinite(L); somme[v] += L[v]; n[v] += 1
        LST = np.where(n >= N_MIN_PIX_EX, somme / np.maximum(n, 1), np.nan)
        ALT = Z_ECO[sl]
        XX, YY = np.meshgrid(xs_e[c0:c1], ys_e[r0:r1])
        pts_ = shapely.points(XX.ravel(), YY.ravel())
        dist = np.where(dans, -shapely.distance(p.boundary, pts_).reshape(h, w), shapely.distance(p, pts_).reshape(h, w))
        t_f, t_a = np.nanmean(LST[fond]), np.nanmean(LST[anneau])
        z_f, z_a = np.nanmean(ALT[fond]), np.nanmean(ALT[anneau])
        grad = (t_a - t_f) / (z_a - z_f) * 100 if z_a - z_f > 1 else np.nan
        vmin, vmax = np.nanpercentile(LST, [2, 98])

        bandes = [('Fond', fond), ('Dépression (hors fond)', dans & ~fond), (f'Tampon 0–{R_TAMPON} m', (dist > 0) & (dist <= R_TAMPON))]
        d0 = R_TAMPON
        while d0 < r_out:
            d1 = min(d0 + 100, r_out); bandes.append((f'Anneau {d0:.0f}–{d1:.0f} m', anneau & (dist > d0) & (dist <= d1))); d0 = d1
        prof_ = pd.DataFrame([{'bande': b, 'pixels': int((mk & np.isfinite(LST)).sum()),
                               'LST': np.nanmean(LST[mk]) if (mk & np.isfinite(LST)).any() else np.nan,
                               'altitude': np.nanmean(ALT[mk]) if mk.any() else np.nan} for b, mk in bandes]).dropna(subset=['LST'])
        ecrire_csv(prof_.round(3), f'exemple_dep{i}_profil.csv')
        ecrire_tif(os.path.join(SORTIES, f'exemple_dep{i}_LST_fin_nuit_hiver.tif'), [LST, n], ['LST_moyenne', 'n_nuits'],
                   T_ECO * Affine.translation(c0, r0), CRS_M)

        def reperes(ax, couleurs):
            contour_pixels(ax, fond, ext, color=couleurs[0], lw=2.2)
            tracer_contour(ax, p, color=couleurs[1], lw=1.4, ls=(0, (6, 4)))
            tracer_contour(ax, p.buffer(R_TAMPON), color=couleurs[1], lw=1.2, ls=(0, (2, 3)))
            tracer_contour(ax, p.buffer(r_out), color=couleurs[2], lw=1.2, ls='-.')
            ii, jj = np.where(exclus)
            ax.plot(ext[0] + (jj + 0.5) * PAS_ECO, ext[3] - (ii + 0.5) * PAS_ECO, 'o', color=couleurs[3], ms=2, alpha=0.7)
            ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_xticks([]); ax.set_yticks([])

        def etiquettes(ax, masque, clair_fonce=True):
            fs = float(np.clip(0.3 * 0.62 * 15 * 72 / max(h, w), 5, 10))
            for a_, b_ in zip(*np.where(masque & np.isfinite(LST))):
                c_ = cmap_ex((LST[a_, b_] - vmin) / (vmax - vmin))[:3]
                clair = clair_fonce and 0.299 * c_[0] + 0.587 * c_[1] + 0.114 * c_[2] > 0.5
                ax.text(ext[0] + (b_ + 0.5) * PAS_ECO, ext[3] - (a_ + 0.5) * PAS_ECO, fr(LST[a_, b_]), ha='center',
                        va='center', fontsize=fs, fontweight='bold' if fond[a_, b_] else 'normal',
                        color='#222' if clair else 'white', zorder=5,
                        path_effects=[pe.withStroke(linewidth=1.8, foreground='white' if clair else 'black', alpha=0.6)])

        texte = (f'Fond : {fond.sum()} pixels, {fr(t_f)} °C\nAnneau : {anneau.sum()} pixels, {fr(t_a)} °C\n'
                 f'Écart fond − bords : {fr(t_f - t_a)} °C\nAltitudes : {z_f:.0f} / {z_a:.0f} m\n'
                 f'Gradient : {fr(grad)} °C/100 m\nModèle (nov.–déc.) : {fr(m["dT_fin_ND"], 2)} °C '
                 f'[{fr(m["dT_fin_ND_IC_bas"], 2)} ; {fr(m["dT_fin_ND_IC_haut"], 2)}]\nNuits : {int(n.max())} au plus par pixel')

        # (a) grille de LST étiquetée (fond complet, anneau échantillonné)
        fig = plt.figure(figsize=(14, 10))
        ax = fig.add_axes([0.03, 0.12, 0.64, 0.8]); cax = fig.add_axes([0.12, 0.06, 0.46, 0.018])
        cm_ = cmap_ex.copy(); cm_.set_bad('#d9d9d9')
        im = ax.imshow(LST, cmap=cm_, vmin=vmin, vmax=vmax, extent=ext, interpolation='nearest')
        grille = np.zeros((h, w), bool); pas_e = max(1, int(np.ceil(max(h, w) / 14))); grille[::pas_e, ::pas_e] = True
        etiquettes(ax, echant(fond) | (anneau & grille))
        reperes(ax, ('k', 'k', 'k', '#666'))
        fig.colorbar(im, cax=cax, orientation='horizontal', label='LST de fin de nuit d\'hiver (°C)')
        ax.set_title(f'Dépression {i} — LST de fin de nuit ({", ".join(NOMS_MOIS[x - 1] for x in MOIS_EXEMPLE)} ; '
                     f'> {PART_MIN_EX:.0%} de la nuit ; neige retirée)', fontsize=12)
        fig.text(0.70, 0.88, texte, fontsize=10, va='top')
        fig.text(0.70, 0.42, 'Trait plein : fond\nTirets : limite de la dépression\nPointillés : fin du tampon\n'
                             'Trait mixte : limite de l\'anneau\nPoints gris : pixels de l\'anneau exclus', fontsize=9, va='top')
        sauver_figure(fig, f'exemple_dep{i}_grille_LST'); plt.show()

        # (b) image satellite + LST en transparence + rayons + profil
        fig = plt.figure(figsize=(16, 11.5))
        ax = fig.add_axes([0.02, 0.13, 0.6, 0.78]); cax = fig.add_axes([0.10, 0.065, 0.44, 0.016])
        axp = fig.add_axes([0.70, 0.55, 0.25, 0.34]); axt = fig.add_axes([0.66, 0.05, 0.32, 0.36]); axt.axis('off')
        ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
        fond_image = False
        if ctx is not None:
            try:
                ctx.add_basemap(ax, crs=CRS_M, source=ctx.providers.Esri.WorldImagery, attribution_size=7, zorder=0,
                attribution="Imagery: Esri, Vantor, Earthstar Geographics, and the GIS User Community")
                fond_image = True
            except Exception as ex:
                print('Image satellite indisponible :', repr(ex)[:100])
        cm_t = cmap_ex.copy(); cm_t.set_bad(alpha=0)
        ax.imshow(LST, cmap=cm_t, vmin=vmin, vmax=vmax, extent=ext, interpolation='nearest',
                  alpha=ALPHA_LST if fond_image else 1, zorder=2)
        ci, cj = np.argwhere(fond).mean(axis=0); rmax = (r_out + np.sqrt(p.area / np.pi)) / PAS_ECO + 1
        rayons = np.zeros((h, w), bool)
        for ang in np.linspace(0, 2 * np.pi, N_RAYONS, endpoint=False):
            for t in np.arange(0, rmax, 1.0):
                a_, b_ = int(round(ci - t * np.sin(ang))), int(round(cj + t * np.cos(ang)))
                if not (0 <= a_ < h and 0 <= b_ < w) or dist[a_, b_] > r_out:
                    break
                rayons[a_, b_] = not exclus[a_, b_]
        etiquettes(ax, echant(fond) | rayons, clair_fonce=False)
        reperes(ax, ('#ffe600', 'white', '#00e5ff', '#ff4fd8'))
        fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap_ex), cax=cax,
                     orientation='horizontal', label='LST de fin de nuit d\'hiver (°C)')
        ax.set_title(f'Dépression {i} — LST de fin de nuit d\'hiver' + (' sur image satellite' if fond_image else '')
                     + f'\n(valeurs le long de {N_RAYONS} rayons) · écart fond − bords : {fr(t_f - t_a)} °C', fontsize=12)
        ax.legend(handles=[Line2D([0], [0], color='#ffe600', lw=2.4, label=f'Fond : {fr(t_f)} °C'),
                           Line2D([0], [0], color='white' if fond_image else 'k', lw=1.4, ls=(0, (6, 4)), label='Limite de la dépression'),
                           Line2D([0], [0], color='white' if fond_image else 'k', lw=1.2, ls=(0, (2, 3)), label=f'Fin du tampon ({R_TAMPON} m)'),
                           Line2D([0], [0], color='#00e5ff', lw=1.2, ls='-.', label=f'Anneau ({r_out:.0f} m) : {fr(t_a)} °C'),
                           Line2D([0], [0], marker='o', color='#ff4fd8', lw=0, ms=4, label='Pixels exclus')],
                  loc='lower left', bbox_to_anchor=(0, 0.035), fontsize=9, framealpha=0.85).set_zorder(6)
        x_ = np.arange(len(prof_))
        axp.plot(x_, prof_['LST'], 'o-', color='#2171b5', lw=2)
        for xi, vi in zip(x_, prof_['LST']):
            axp.annotate(fr(vi), (xi, vi), textcoords='offset points', xytext=(0, 7), ha='center', fontsize=8, color='#2171b5')
        axp.set_ylabel('LST moyenne (°C)', color='#2171b5')
        axa = axp.twinx(); axa.plot(x_, prof_['altitude'], 's--', color='#7f4f24', lw=1.2, ms=4)
        axa.set_ylabel('Altitude moyenne (m)', color='#7f4f24')
        axp.set_xticks(x_); axp.set_xticklabels([b.replace(' ', '\n', 1) for b in prof_['bande']], rotation=55, ha='right', fontsize=8)
        axp.set_title('Profil du fond vers les bords (LST en bleu, altitude en brun)', fontsize=10); axp.grid(alpha=0.3)
        tb = axt.table(cellText=[[b, str(k_), fr(t_), f'{z_:.0f}'] for b, k_, t_, z_ in prof_.itertuples(index=False)],
                       colLabels=['Bande', 'Pixels', 'LST (°C)', 'Altitude (m)'], loc='upper center', cellLoc='center')
        tb.auto_set_font_size(False); tb.set_fontsize(9); tb.auto_set_column_width(col=list(range(4))); tb.scale(1, 1.35)
        axt.text(0.5, -0.04, f'Gradient fond → bords : {fr(grad)} °C/100 m', fontsize=10, transform=axt.transAxes,
                 ha='center', va='top')
        sauver_figure(fig, f'exemple_dep{i}_profil_satellite'); plt.show()

In [ ]:
# ============================================================
# 20. EXEMPLES DE CUVETTES : SÉRIE NUIT PAR NUIT ET CYCLE SAISONNIER (début et fin de nuit)
# ============================================================
# Réf. : Göttsche & Olesen (2001, RSE), Duan et al. (2012, RSE) — part de nuit/jour.
# Repose sur le tableau des passages (cellule 15) : mêmes fond, anneau et retrait de la neige que le modèle.
# Points mensuels avec IC 95 % (symbole vide si < 5 nuits) ; trait : moyenne glissante sur 3 mois.
if FIGURES_EXEMPLES:
    import matplotlib.dates as mdates
    for i in EX:
        s = Rn[Rn['id_dep'] == i].sort_values('utc').copy()
        s['mois'] = s['utc'].dt.month
        ecrire_csv(s.round({c: 3 for c in s.select_dtypes('number').columns}), f'exemple_dep{i}_serie_nuits.csv')
        fig = plt.figure(figsize=(16, 9))
        ax1 = fig.add_axes([0.05, 0.56, 0.55, 0.36]); ax2 = fig.add_axes([0.05, 0.08, 0.55, 0.36], sharex=ax1)
        ax3 = fig.add_axes([0.70, 0.08, 0.28, 0.84]); cax1 = fig.add_axes([0.61, 0.56, 0.008, 0.36])
        ax1.plot(s['utc'], s['T_bords'], 'o', color='#bdbdbd', ms=4, label='bords (anneau)')
        sc = ax1.scatter(s['utc'], s['T_fond'], c=s['part_nuit'], cmap='viridis', vmin=0, vmax=1, s=24,
                         edgecolor='k', linewidth=0.3, label='fond', zorder=3)
        ax1.set_ylabel('LST moyenne (°C)'); ax1.legend(loc='upper left', fontsize=9); ax1.grid(alpha=0.3)
        fig.colorbar(sc, cax=cax1, label='Part de la nuit écoulée')
        ax2.bar(s['utc'], s['dT'], width=6, color=np.where(s['dT'] < 0, '#2171b5', '#fd8d3c'))
        ax2.axhline(0, color='k', lw=0.8); ax2.set_ylabel('Écart fond − bords (°C)'); ax2.grid(alpha=0.3)
        ax2.xaxis.set_major_locator(mdates.YearLocator()); ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        for nom, msk, c in [('début de nuit (< 0,4)', s['part_nuit'] < 0.4, '#7b3294'),
                            ('fin de nuit (> 0,6)', s['part_nuit'] > 0.6, '#d95f02')]:
            q = s[msk]
            gm = q.groupby('mois')['dT'].agg(['mean', 'sem', 'size']).reindex(range(1, 13))
            for mm, r in gm.iterrows():
                if np.isfinite(r['mean']):
                    ax3.errorbar(mm, r['mean'], yerr=1.96 * (r['sem'] if r['size'] > 1 else 0), fmt='o', color=c,
                                 mfc=c if r['size'] >= 5 else 'white', capsize=3)
            ax3.plot(range(1, 13), [q[q['mois'].isin([(m_ - 2) % 12 + 1, m_, m_ % 12 + 1])]['dT'].mean()
                                    for m_ in range(1, 13)], '-', color=c, lw=2, label=nom)
        ax3.axhline(0, color='k', lw=0.8); ax3.set_xticks(range(1, 13)); ax3.set_xticklabels(NOMS_MOIS, rotation=45)
        ax3.set_ylabel('Écart fond − bords (°C)'); ax3.legend(fontsize=9); ax3.grid(alpha=0.3)
        ax3.set_title('Cycle saisonnier (IC 95 % ; vide : < 5 nuits)', fontsize=10)
        fig.suptitle(f'Dépression {i} — {len(s)} nuits solaires (bleu : fond plus froid que ses bords)', fontsize=12)
        sauver_figure(fig, f'exemple_dep{i}_serie'); plt.show()

## E. Paquet de données pour Zenodo (mode auteur)

In [ ]:
# ============================================================
# 21. PRÉPARATION DU PAQUET DE DONNÉES (archive à déposer sur Zenodo)
# ============================================================
# Rassemble les entrées et le cube sous les noms attendus par le mode autonome, avec une empreinte SHA-256
# par fichier et un fichier LISEZMOI ; l'archive est écrite dans sorties/.
if MODE == 'auteur' and PREPARER_PAQUET:
    racine = os.path.join(TMP, 'CAPS_ECOSTRESS_donnees'); shutil.rmtree(racine, ignore_errors=True)
    ent, cub = os.path.join(racine, 'entrees'), os.path.join(racine, 'cube_ECOSTRESS')
    os.makedirs(ent); os.makedirs(cub)
    copies = {'zet': 'ZET_3forets.geojson', 'fabdem': 'FABDEM_30m_marge5km.tif', 'cuv_tpi': 'cuvettes_TPI_30m.tif',
              'microrefuges': 'microrefuges_MODIS_1km.csv', 'foret': 'foret_WorldCover2021_70m.tif',
              'neige': 'neige_MOD10A1F_grille_ECO.npz'}
    for cle, nom in copies.items():
        if os.path.exists(F[cle]):
            shutil.copy(F[cle], os.path.join(ent, nom))
    for f in fichiers_cube() + [os.path.join(DIR_CUBE, 'index_passages.csv')]:
        shutil.copy(f, cub)
    with open(os.path.join(racine, 'LISEZMOI.txt'), 'w', encoding='utf-8') as fh:
        fh.write('Données du notebook CAPS_ECOSTRESS (poches d\'air froid, Moyen Atlas central, Maroc).\n'
                 'Grille : Merchich / Nord Maroc (équivalent EPSG:26191), 70 m.\n'
                 'cube_ECOSTRESS : LST ECOSTRESS ECO_L2T_LSTE V002 (NASA LP DAAC) par passage, drapeau neige MOD10A1F V061.\n'
                 'entrees/FABDEM_30m_marge5km.tif : extrait de FABDEM (Hawker et al., 2022), licence CC BY-NC-SA 4.0.\n'
                 'entrees/foret_WorldCover2021_70m.tif : dérivé d\'ESA WorldCover 2021 v200, licence CC BY 4.0.\n'
                 'Citation : Chahbar, K. & Nejjari, A. (2026). ECOSTRESS nocturnal land surface temperature data\n'
                 'cube and cold-air pool outputs, Central Middle Atlas (Morocco), 2019–2026. Zenodo.\n'
                 'https://doi.org/10.5281/zenodo.23472840.\n')
    lignes = []
    for d_, _, fs in os.walk(racine):
        for f in sorted(fs):
            ch = os.path.join(d_, f); h_ = hashlib.sha256()
            with open(ch, 'rb') as fh:
                for bloc in iter(lambda: fh.read(1 << 20), b''):
                    h_.update(bloc)
            lignes.append({'fichier': os.path.relpath(ch, racine), 'octets': os.path.getsize(ch), 'sha256': h_.hexdigest()})
    pd.DataFrame(lignes).to_csv(os.path.join(racine, 'MANIFESTE_sha256.csv'), index=False)
    arch = shutil.make_archive(os.path.join(TMP, 'CAPS_ECOSTRESS_donnees'), 'zip', TMP, 'CAPS_ECOSTRESS_donnees')
    copie_atomique(arch, os.path.join(SORTIES, 'CAPS_ECOSTRESS_donnees.zip'))
    print(f'Archive : {os.path.join(SORTIES, "CAPS_ECOSTRESS_donnees.zip")} '
          f'({os.path.getsize(arch) / 1e6:.0f} Mo, {len(lignes)} fichiers)')
else:
    print('Section E ignorée.')

## Références

Bibliographie complète des méthodes et outils du pipeline. Les lignes `# Réf. :` en tête de chaque section renvoient ici.

### Produits et données
- Hook, S. & Hulley, G. (2022). *ECOSTRESS Tiled Land Surface Temperature and Emissivity Instantaneous L2 Global 70 m v002*. NASA LP DAAC. https://doi.org/10.5067/ECOSTRESS/ECO_L2T_LSTE.002
- Hulley, G. & Freepartner, R. (2024). *ECOSTRESS Mission Level 2 Product User Guide v002*. NASA JPL.
- Hulley, G.C., Göttsche, F.M., Rivera, G., Hook, S.J., Freepartner, R.J., Martin, M.A., Cawse-Nicholson, K. & Johnson, W.R. (2022). *Validation and Quality Assessment of the ECOSTRESS Level-2 Land Surface Temperature and Emissivity Product*. IEEE Transactions on Geoscience and Remote Sensing 60, art. n° 5000523, 1–23. https://doi.org/10.1109/TGRS.2021.3079879
- Fisher, J.B., Lee, B., Purdy, A.J., Halverson, G.H., Dohlen, M.B. et al. (59 auteurs) (2020). *ECOSTRESS: NASA's Next Generation Mission to Measure Evapotranspiration From the International Space Station*. Water Resources Research 56(4), e2019WR026058. https://doi.org/10.1029/2019WR026058
- Riggs, G.A., Hall, D.K. & Roman, M.O. (2019). *MODIS Snow Products Collection 6.1 User Guide*. NSIDC ; MOD10A1F v061 : https://doi.org/10.5067/MODIS/MOD10A1F.061
- Hall, D.K., Riggs, G.A., Salomonson, V.V., DiGirolamo, N.E. & Bayr, K.J. (2002). *MODIS snow-cover products*. Remote Sensing of Environment 83, 181–194. https://doi.org/10.1016/S0034-4257(02)00095-0
- Hall, D.K., Riggs, G.A., Foster, J.L. & Kumar, S.V. (2010). *Validation of the MODIS cloud-gap-filled snow-cover product*. Remote Sensing of Environment 114, 496–503. https://doi.org/10.1016/j.rse.2009.10.007
- Hawker, L., Uhe, P., Paulo, L., Sosa, J., Savage, J., Sampson, C. & Neal, J. (2022). *FABDEM v1-0*. Zenodo. https://doi.org/10.5281/zenodo.5777913 (CC BY-NC-SA 4.0)
- Zanaga, D. et al. (2022). *ESA WorldCover 10 m 2021 v200*. Zenodo. https://doi.org/10.5281/zenodo.7254221 (CC BY 4.0)
- Esri. *World Imagery* (fond cartographique, via contextily ; Esri Master License Agreement) : crédits affichés sur chaque carte, conformément aux règles d'Esri pour les publications.

### Traitement d'images, géomorphologie et cycles thermiques
- Clinton, N. & Gong, P. (2013). *MODIS detected surface urban heat islands and sinks: Global locations and controls*. Remote Sensing of Environment 134, 294–304. https://doi.org/10.1016/j.rse.2013.03.008
- Göttsche, F.-M. & Olesen, F.S. (2001). *Modelling of diurnal cycles of brightness temperature extracted from METEOSAT data*. Remote Sensing of Environment 76(3), 337–348. https://doi.org/10.1016/S0034-4257(00)00214-5
- Duan, S.-B., Li, Z.-L., Wang, N., Wu, H. & Tang, B.-H. (2012). *Evaluation of six land-surface diurnal temperature cycle models using clear-sky in situ and satellite data*. Remote Sensing of Environment 124, 15–25. https://doi.org/10.1016/j.rse.2012.04.016
- Vincent, L. (1993). *Morphological grayscale reconstruction in image analysis: applications and efficient algorithms*. IEEE Transactions on Image Processing 2(2), 176–201. https://doi.org/10.1109/83.217222
- O'Callaghan, J.F. & Mark, D.M. (1984). *The extraction of drainage networks from digital elevation data*. Computer Vision, Graphics, and Image Processing 28(3), 323–344. https://doi.org/10.1016/0734-189X(84)80011-0
- Barnes, R., Lehman, C. & Mulla, D. (2014a). *Priority-flood: An optimal depression-filling and watershed-labeling algorithm for digital elevation models*. Computers & Geosciences 62, 117–127. https://doi.org/10.1016/j.cageo.2013.04.024
- Barnes, R., Lehman, C. & Mulla, D. (2014b). *An efficient assignment of drainage direction over flat surfaces in raster digital elevation models*. Computers & Geosciences 62, 128–135. https://doi.org/10.1016/j.cageo.2013.01.009
- Bartos, M. (2020). *pysheds: simple and fast watershed delineation in python*. Zenodo. https://doi.org/10.5281/zenodo.3822494

### Pièges à froid et climat de terrain
- Clements, C.B., Whiteman, C.D. & Horel, J.D. (2003). *Cold-air-pool structure and evolution in a mountain basin: Peter Sinks, Utah*. Journal of Applied Meteorology 42(6), 752–768. https://doi.org/10.1175/1520-0450(2003)042<0752:CSAEIA>2.0.CO;2
- Whiteman, C.D., Haiden, T., Pospichal, B., Eisenbach, S. & Steinacker, R. (2004). *Minimum temperatures, diurnal temperature ranges, and temperature inversions in limestone sinkholes of different sizes and shapes*. Journal of Applied Meteorology 43(8), 1224–1236. https://doi.org/10.1175/1520-0450(2004)043<1224:MTDTRA>2.0.CO;2
- Steinacker, R., Whiteman, C.D., Dorninger, M., Pospichal, B., Eisenbach, S., Holzer, A.M., Weihs, P., Mursch-Radlgruber, E. & Baumann, K. (2007). *A sinkhole field experiment in the Eastern Alps*. Bulletin of the American Meteorological Society 88(5), 701–716. https://doi.org/10.1175/BAMS-88-5-701
- Dobrowski, S.Z. (2011). *A climatic basis for microrefugia: the influence of terrain on climate*. Global Change Biology 17(2), 1022–1035. https://doi.org/10.1111/j.1365-2486.2010.02263.x
- Pastore, M.A., Classen, A.T., D'Amato, A.W., English, M.E., Rand, K., Foster, J.R. & Adair, E.C. (2024). *Frequent and strong cold-air pooling drives temperate forest composition*. Ecology and Evolution 14, e11126. https://doi.org/10.1002/ece3.11126
- Noad, N.C. & Bonnaventure, P.P. (2022). *Surface temperature inversion characteristics in dissimilar valleys, Yukon Canada*. Arctic Science 8, 1320–1339. https://doi.org/10.1139/as-2021-0048 (méthode des indicateurs de CAP reprise par Pastore et al. 2024).

### Statistiques
- Mann, H.B. & Whitney, D.R. (1947). *On a test of whether one of two random variables is stochastically larger than the other*. Annals of Mathematical Statistics 18, 50–60. https://doi.org/10.1214/aoms/1177730491
- Wilcoxon, F. (1945). *Individual comparisons by ranking methods*. Biometrics Bulletin 1(6), 80–83. https://doi.org/10.2307/3001968
- Kerby, D.S. (2014). *The Simple Difference Formula: An Approach to Teaching Nonparametric Correlation*. Comprehensive Psychology 3. https://doi.org/10.2466/11.it.3.1
- Spearman, C. (1904). *The proof and measurement of association between two things*. American Journal of Psychology 15(1), 72–101. https://doi.org/10.2307/1422010
- Hanley, J.A. & McNeil, B.J. (1982). *The meaning and use of the area under a receiver operating characteristic (ROC) curve*. Radiology 143(1), 29–36. https://doi.org/10.1148/radiology.143.1.7063747
- Cohen, J. (1960). *A coefficient of agreement for nominal scales*. Educational and Psychological Measurement 20(1), 37–46. https://doi.org/10.1177/001316446002000104
- Barnett, A.G., Van Der Pols, J.C. & Dobson, A.J. (2005). *Regression to the mean: what it is and how to deal with it*. International Journal of Epidemiology 34(1), 215–220. https://doi.org/10.1093/ije/dyh299
- Efron, B. & Tibshirani, R. (1986). *Bootstrap methods for standard errors, confidence intervals, and other measures of statistical accuracy*. Statistical Science 1(1), 54–75. https://doi.org/10.1214/ss/1177013815
- Efron, B. & Tibshirani, R.J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- Good, P. (2005). *Permutation, Parametric and Bootstrap Tests of Hypotheses* (3e éd.). Springer.
- Jönsson, P. & Eklundh, L. (2002). *Seasonality extraction by function fitting to time-series of satellite sensor data*. IEEE Transactions on Geoscience and Remote Sensing 40(8), 1824–1832. https://doi.org/10.1109/tgrs.2002.802519
- Jönsson, P. & Eklundh, L. (2004). *TIMESAT—a program for analyzing time-series of satellite sensor data*. Computers & Geosciences 30(8), 833–845. https://doi.org/10.1016/j.cageo.2004.05.006

### Conventions et logiciels
- Eaton, B., Gregory, J., Drach, B., Taylor, K., Hankin, S. et al. *NetCDF Climate and Forecast (CF) Metadata Conventions* — CF-1.8. https://cfconventions.org
- NOAA. *Solar Calculation Details*. https://gml.noaa.gov/grad/solcalc/ ; Meeus, J. (1991). *Astronomical Algorithms*. Willmann-Bell.
- Harris, C.R. et al. (2020). *Array programming with NumPy*. Nature 585, 357–362. https://doi.org/10.1038/s41586-020-2649-2
- Virtanen, P. et al. (2020). *SciPy 1.0: fundamental algorithms for scientific computing in Python*. Nature Methods 17, 261–272. https://doi.org/10.1038/s41592-019-0686-2
- pandas development team. *pandas*. Zenodo. https://doi.org/10.5281/zenodo.3509134
- Hunter, J.D. (2007). *Matplotlib: A 2D graphics environment*. Computing in Science & Engineering 9(3), 90–95. https://doi.org/10.1109/MCSE.2007.55
- van der Walt, S. et al. (2014). *scikit-image: image processing in Python*. PeerJ 2, e453. https://doi.org/10.7717/peerj.453
- Jordahl, K. et al. *geopandas*. Zenodo. https://doi.org/10.5281/zenodo.594445
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine learning in Python*. Journal of Machine Learning Research 12, 2825–2830.
- Gorelick, N. et al. (2017). *Google Earth Engine: Planetary-scale geospatial analysis for everyone*. Remote Sensing of Environment 202, 18–27. https://doi.org/10.1016/j.rse.2017.06.031

### Notes de prudence
1. **LST ≠ air.** Pastore et al. (2024) et Whiteman et al. (2004) mesurent la température de l'air (1,4–2,2 m) ; ce pipeline mesure la température de surface (LST). L'interprétation en termes de « piège à froid » repose sur la LST — le rapport air–surface peut différer selon le couvert et l'humidité.
2. **SVF.** Whiteman et al. (2004) montrent que le facteur de vue du ciel (SVF) est le premier contrôle du refroidissement nocturne : la doline de 54 m (Gruenloch, aire de drainage totale 2,12 km²) et celle de 26 m (Seekopfalm, ~12 % de cette aire selon le résumé de l'article) atteignent des minima quasi identiques (−28,5 / −27,8 °C, nuit de décembre) malgré des tailles très différentes. Cela nuance les déterminants « surface » et « dénivelé » de la section 16. Le pipeline ne calcule pas le SVF ; ce facteur n'est donc pas pris en compte parmi les déterminants.

